In [9]:
import numpy as np
import pandas as pd

from pyboat import ensemble_measures as em
from pyboat import WAnalyzer, ssg
from pyboat import plotting as pl

import scipy.signal
import pywt

import matplotlib.pyplot as plt

import bokeh_catplot
from bokeh.plotting import figure, show
from bokeh.models import FactorRange, Button, ColumnDataSource, HoverTool, TapTool, Slider, Select, Span, BoxZoomTool

import bokeh.io

import iqplot

bokeh.io.output_notebook()

Loading BokehJS ...

In [10]:
cd H:/PROJECTS-03/Pablo/oscillating/ppf021_analysis_median_corr/

H:\PROJECTS-03\Pablo\oscillating\ppf021_analysis_median_corr


In [11]:
notebook_url = 'localhost:8888'

In [12]:
def analyze_signal(signals, df_fouriers, wAn, save_path=None, plot=False):
    for signal in signals.columns:
        signals[signal] = wAn.sinc_detrend(signals[signal], T_c = 275)
        wAn.compute_spectrum(signals[signal], do_plot=plot)
        if plot == True:
            plt.title(signal)
        df_fouriers[signal] = wAn.get_averaged_spectrum()
        if save_path is not None:
            plt.savefig(fname=save_path+signal)
    return df_fouriers

In [13]:
def plot_fourier(freq_domain, signal, peak, y_value):
    x = freq_domain
    y = signal
    plt.plot(x, y)
    plt.xlim(np.min(x), np.max(x))
    plt.xlabel('Freq (min)')
    plt.ylabel('Power')
    plt.plot(peak, y_value, marker="x", markersize=5, markeredgecolor="red", markerfacecolor="red")    

In [14]:
def process_peaks(df_fouriers, freq_domain, cut_off=130, peak_prom=0.50):
    max_peak = []
    name_signal = []
    non_processed = []
    for f_signal in df_fouriers.columns:
        try:
            # peak = freq_domain[np.max(scipy.signal.find_peaks(df_fouriers[f_signal], prominence=peak_prom, height=1, width=5)[0])]
            peak = freq_domain[np.min(scipy.signal.find_peaks(df_fouriers[f_signal], prominence=peak_prom, height=1)[0])]
            if peak < cut_off: 
                max_peak.append(peak)
                name_signal.append(f_signal)
        except ValueError:
            print(f'Cannot find peak for {f_signal}')
            non_processed.append(f_signal)
    return np.array(max_peak, dtype=np.float32), name_signal, non_processed

In [15]:
def check_peaks(signals, peaks, df_fouriers, agg=False, med_mean = False, title=None):
    for signal, peak in zip(signals, peaks):
        freq_domain = df_fouriers.index
        closest_value = min(df_fouriers_control.index, key=lambda x: abs(peak - x))
        y_value = df_fouriers.loc[closest_value, signal]
        plot_fourier(freq_domain, df_fouriers[signal], peak, y_value)
        if agg==True:
            plt.ylim(0,8)
            plt.yticks(ticks=range(0, 13, 1), labels=map(str, range(0, 13 , 1)))
        else:
            plt.title(signal)
            plt.show()

    if agg == True:
        plt.title(title)
        if med_mean==True:
            plt.axvline(x = np.mean(peaks), color ='b')
            plt.axvline(x = np.median(peaks), color ='r')
            plt.show()
        else:
            plt.show()

In [16]:
def normalize_df(dataframe):
    normalized_dataframe = dataframe.copy()
    for column in dataframe.columns.values:
        normalized_dataframe[column] = (normalized_dataframe[column] - normalized_dataframe[column].min()) / (normalized_dataframe[column].max() - normalized_dataframe[column].min())  
    return normalized_dataframe

In [18]:
df_total = pd.read_csv('./valid_signals_ppf021_bs_median', index_col=0)

In [20]:
df_total

,ppf021_xy001_0,ppf021_xy004_0,ppf021_xy005_0,ppf021_xy006_0,ppf021_xy007_0,ppf021_xy008_0,ppf021_xy010_0,ppf021_xy011_0,ppf021_xy012_1,ppf021_xy013_0,...,ppf021_xy097_0,ppf021_xy098_0,ppf021_xy099_0,ppf021_xy102_0,ppf021_xy103_0,ppf021_xy104_0,ppf021_xy105_0,ppf021_xy107_0,ppf021_xy108_0,ppf021_xy108_1
0,91.873883,50.829587,22.839226,31.950820,65.159537,92.487701,27.365672,43.669065,28.191723,5.776879,...,118.589124,5.482921,18.390572,35.721491,18.113577,154.502481,17.023029,23.610514,50.025907,NaN
1,178.274662,58.515997,22.879112,25.113028,87.477547,105.922777,27.960000,38.897959,21.784641,14.317048,...,204.514626,26.548428,37.578054,48.280702,22.832237,77.892655,17.775758,20.166494,42.437349,44.762931
2,328.982558,154.519333,26.633609,27.912429,51.718808,116.369820,34.025400,26.297647,11.068235,13.133745,...,223.984547,35.083426,24.634801,50.595583,14.875661,41.976608,10.977605,24.944504,36.583120,67.409864
3,334.455954,112.176833,12.400550,36.521696,25.779284,127.630016,20.890088,7.292571,7.174603,2.285223,...,163.674699,86.572553,28.673469,55.087705,8.551247,30.983412,17.692656,27.153252,24.992560,50.708713
4,298.308066,78.035040,10.114663,109.237023,65.847358,51.891210,13.155626,12.758192,11.435776,4.407725,...,86.100585,58.165327,11.134680,42.078905,5.881694,28.991898,38.913823,17.995495,19.587940,73.401062
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72,148.478082,43.547076,13.669303,50.443089,3.898598,11.490652,37.248497,16.181943,16.438723,16.832402,...,130.915835,28.010432,25.278320,31.325545,19.120240,34.533432,11.524784,16.398158,46.657534,42.798551
73,156.790503,52.038320,12.153325,51.984756,14.993281,9.648637,35.183283,10.873950,16.555215,15.396254,...,143.862117,20.246677,30.997625,32.143417,25.258621,29.294437,12.969474,24.356808,36.687606,44.544118
74,166.850914,46.908492,6.069994,46.114428,17.407530,9.539211,32.973790,7.692722,12.784387,14.967789,...,139.785814,21.933803,41.861945,40.712702,23.637149,27.574405,15.103049,40.729456,41.887372,48.162554
75,163.977120,43.208155,9.564286,51.122288,9.457014,8.333775,35.484064,13.251389,7.346959,20.953010,...,148.917024,16.944858,29.677189,29.723765,12.068359,35.808480,13.342827,23.909091,41.177554,35.285068


In [19]:
all_signals_norm = normalize_df(df_total)

In [26]:
df_total = df_total.iloc[:,1:]

In [16]:
df_total = df_total.dropna(axis=1)

In [17]:
all_signals_norm = normalize_df(df_total)

In [21]:
all_signals_norm

,ppf021_xy001_0,ppf021_xy004_0,ppf021_xy005_0,ppf021_xy006_0,ppf021_xy007_0,ppf021_xy008_0,ppf021_xy010_0,ppf021_xy011_0,ppf021_xy012_1,ppf021_xy013_0,...,ppf021_xy097_0,ppf021_xy098_0,ppf021_xy099_0,ppf021_xy102_0,ppf021_xy103_0,ppf021_xy104_0,ppf021_xy105_0,ppf021_xy107_0,ppf021_xy108_0,ppf021_xy108_1
0,0.100502,0.044921,0.014477,0.009688,0.067606,0.077501,0.012649,0.056877,0.076489,0.004399,...,0.081613,0.000000,0.008180,0.026740,0.037047,0.209098,0.028289,0.015852,0.088207,NaN
1,0.272395,0.056772,0.014511,0.004121,0.092223,0.089515,0.013178,0.050606,0.056913,0.014482,...,0.160084,0.044270,0.029812,0.053800,0.048931,0.102633,0.029550,0.012403,0.069795,0.018659
2,0.572226,0.204797,0.017688,0.006400,0.052780,0.098856,0.018577,0.034045,0.024171,0.013085,...,0.177865,0.062207,0.015220,0.058788,0.028893,0.052721,0.018168,0.017187,0.055591,0.038964
3,0.583115,0.139510,0.005645,0.013409,0.024169,0.108925,0.006885,0.009067,0.012275,0.000277,...,0.122787,0.170414,0.019773,0.068467,0.012964,0.037443,0.029410,0.019399,0.027470,0.023990
4,0.511199,0.086868,0.003710,0.072602,0.068365,0.041199,0.000000,0.016250,0.025294,0.002783,...,0.051943,0.110715,0.000000,0.040438,0.006241,0.034676,0.064941,0.010229,0.014356,0.044335
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72,0.213115,0.033692,0.006718,0.024741,0.000034,0.005073,0.021446,0.020750,0.040580,0.017452,...,0.092870,0.047343,0.015945,0.017269,0.039583,0.042377,0.019084,0.008629,0.080035,0.016898
73,0.229652,0.046785,0.005435,0.025996,0.012271,0.003425,0.019608,0.013774,0.040936,0.015756,...,0.104693,0.031027,0.022393,0.019031,0.055042,0.035096,0.021503,0.016599,0.055845,0.018463
74,0.249667,0.038875,0.000288,0.021218,0.014934,0.003328,0.017641,0.009592,0.029415,0.015250,...,0.100971,0.034572,0.034641,0.037494,0.050958,0.032706,0.025075,0.032994,0.068461,0.021707
75,0.243950,0.033170,0.003245,0.025294,0.006165,0.002250,0.019876,0.016898,0.012802,0.022317,...,0.109310,0.024088,0.020904,0.013818,0.021822,0.044149,0.022128,0.016151,0.066739,0.010161


In [36]:
all_signals_norm.dropna(axis=1, inplace=True)

In [38]:
len(all_signals_norm)

81

In [27]:
p = bokeh_catplot.strip(
    data=df_total,
    cats='Condition',
    val='Max. power period (min)',
    horizontal=False,
    jitter=True,
    height=1000,
    width=1400,
    tooltips=[('position', '@{Cell_position}'), ('period', '@{Max. power period (min)}')]
    
)

p = bokeh_catplot.box(
    data=df_total,
    cats='Condition',
    val='Max. power period (min)',
    horizontal=False,
    whisker_caps=True,
    display_points=False,
    box_kwargs=dict(fill_color=None, line_color='gray'),
    median_kwargs=dict(line_color='gray'),
    whisker_kwargs=dict(line_color='gray'),
    height=1000,
    width=1400,
    p=p,
)

bokeh.io.show(p)

C:\Users\pperez\.conda\envs\devbio-napari-cupy\lib\site-packages\bokeh\io\notebook.py:487: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  publish_display_data(data, metadata, source, transient=transient, **kwargs)


In [20]:
df_total = df_total.drop(df_total.loc[df_total['Cell_position'] == 'ppf005_xy031_0',:].index)

In [21]:
df_total = df_total.drop(df_total.loc[df_total['Cell_position'] == 'ppf005_xy107_1',:].index)

In [22]:
df_total = df_total.drop(df_total.loc[df_total['Cell_position'] == 'ppf005_xy116_0',:].index)

In [23]:
df_total = df_total.drop(df_total.loc[df_total['Cell_position'] == 'ppf005_xy102_0',:].index)

In [24]:
df_total = df_total.drop(df_total.loc[df_total['Cell_position'] == 'ppf005_xy105_0',:].index)
df_total = df_total.drop(df_total.loc[df_total['Cell_position'] == 'ppf005_xy105_2',:].index)
df_total = df_total.drop(df_total.loc[df_total['Cell_position'] == 'ppf005_xy106_0',:].index)

In [25]:
df_total = df_total.drop(df_total.loc[df_total['Cell_position'] == 'ppf005_xy058_0',:].index)

In [26]:
df_total = df_total.drop(df_total.loc[df_total['Cell_position'] == 'ppf005_xy120_0',:].index)

In [335]:
from bokeh.transform import jitter
from io import BytesIO
from bokeh.models import Div, Legend, ResetTool
import base64

df_plot = df_total.copy(deep=True)
categories = df_plot['Condition'].unique()
colors = [bokeh.palettes.d3['Category20'][5][0],
        bokeh.palettes.d3['Category20'][5][-1],
        bokeh.palettes.d3['Category20'][5][-3]]



category_colors = dict(zip(categories, colors))
df_plot['color'] = df_plot['Condition'].map(category_colors)
source = ColumnDataSource(df_plot)
p = figure(width=900, height=800, x_range=df_total['Condition'].unique(),
          tools=[TapTool(), BoxZoomTool(), ResetTool()])



# categories = 
# experiments = 
# p.circle(x='Max. power period (min)', y=jitter('Condition', width=0.6, range=p.y_range),  source=source, alpha=1.0, legend_field='Condition', color='color', nonselection_alpha=1.0)
p.circle(x=jitter('Condition', width=0.20, range=p.x_range), y='Max. power period (min)',  source=source, alpha=1.0, legend_group='Condition', color='color', nonselection_alpha=1.0)


image_div = Div(text='', width=800, height=800)

def tapping(attr, old, new):
    try:
        selected_index = source.selected.indices[0]
        #print(f"Clicked on point at index {selected_index}")
        
        # Display an image associated with the point index
        image_path = './figures/'+df_total.iloc[selected_index, -1]+'.png'
    
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")

        # Update the content of the Div widget with the new image
        image_div.text = f'<img src="data:image/png;base64,{image_data}" width="800" height="800">'
    except IndexError:
        pass

    
# Change the selected index by Tapping
source.selected.on_change('indices', tapping)


# Change font size of axes
p.xaxis.major_label_text_font_size = '14pt'
p.yaxis.major_label_text_font_size = '14pt'
p.xaxis.axis_label = 'Condition'
p.yaxis.axis_label = 'Period (min)'

#Change title
p.title.text = 'Period vs CHX concentration'
p.title.text_font_size = '16pt'
p.title.align = 'center'

norm_layout = bokeh.layouts.row(
    p,
    bokeh.models.Spacer(height=15),
    image_div
)
# Add layout to the current document
def norm_app(doc):
    doc.add_root(norm_layout)

notebook_url = 'localhost:8888'
bokeh.io.show(norm_app, notebook_url=notebook_url)

#output_file('interactive_categorical_strip_plot.html')
#show(p)

INFO:bokeh.server.server:Starting Bokeh server version 2.4.3 (running on Tornado 6.2)
INFO:bokeh.server.tornado:User authentication hooks NOT provided (default user enabled)
C:\Users\pperez\.conda\envs\devbio-napari-cupy\lib\site-packages\bokeh\io\notebook.py:487: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  publish_display_data(data, metadata, source, transient=transient, **kwargs)


INFO:tornado.access:200 GET /autoload.js?bokeh-autoload-element=139652&bokeh-absolute-url=http://localhost:53475&resources=none (::1) 26.04ms
INFO:tornado.access:101 GET /ws (::1) 1.00ms
INFO:bokeh.server.views.ws:WebSocket connection opened
INFO:bokeh.server.views.ws:ServerConnection created


In [34]:
df_total.loc

In [35]:
df_total['Condition'].unique()

array(['Control', '40 nM', '80 nM'], dtype=object)

In [30]:
# Create a Bokeh figurunique= 
unique = figure(plot_width=600, plot_height=400, tools=[HoverTool(), TapTool()],
           tooltips=[("Index", "$index"), ("(x, y)", "(@x, @y)"), ("Image", "@image_url")])

# Scatter plot with different colors for each category
p.circle('x', 'y', size=15, source=source, color='category', legend_field='category')
def hovering(attr, old, new):
    selected_index = source.selected.indices[0]
    print(f"Clicked on point at index {selected_index}")
    # You can perform actions based on the selected point index, e.g., display an image
    plt.show('./figures/'+source.iloc[selected_index, -1])

source.selected.on_change('indices', hovering)

output_file('interactive_categorical_strip_plot.html')
show(p)

RuntimeError: _pending_writes should be non-None when we have a document lock, and we should have the lock when the document changes

In [31]:
p = bokeh_catplot.strip(
    data=df_total,
    cats='Condition',
    val='Max. power period (min)',
    horizontal=False,
    jitter=True,
    height=650,
    width=850,
    tooltips=[('Position', '@{Cell_position}'),
             ('Period', '@{Max. power period (min)}')]
    
)

p = bokeh_catplot.box(
    data=df_total,
    cats='Condition',
    val='Max. power period (min)',
    horizontal=False,
    whisker_caps=True,
    display_points=False,
    box_kwargs=dict(fill_color=None, line_color='gray'),
    median_kwargs=dict(line_color='gray'),
    whisker_kwargs=dict(line_color='gray'),
    height=650,
    width=850,
    p=p,
)

bokeh.io.show(p)

C:\Users\pperez\.conda\envs\devbio-napari-cupy\lib\site-packages\bokeh\io\notebook.py:487: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  publish_display_data(data, metadata, source, transient=transient, **kwargs)


In [32]:
p = iqplot.ecdf(
    data=df_total,
    q="Max. power period (min)",
    cats='Condition',
    kind='colored',
    tooltips=[('Position', '@{Cell_position}'),
             ('Period', '@{Max. power period (min)}')],
)
bokeh.io.show(p)

C:\Users\pperez\.conda\envs\devbio-napari-cupy\lib\site-packages\bokeh\io\notebook.py:487: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  publish_display_data(data, metadata, source, transient=transient, **kwargs)


In [33]:
p = iqplot.ecdf(
    data=df_total.loc[df_total['Condition'] == 'Control',:],
    q="Max. power period (min)",
    cats='Condition',
    kind='colored',
    palette=bokeh.palettes.d3['Category20'][5][0],
    tooltips=[('Position', '@{Cell_position}'),
             ('Period', '@{Max. power period (min)}')],
    height=800,
    width=800
)

p = iqplot.ecdf(
    data=df_total.loc[df_total['Condition'] == '40 nM',:],
    q="Max. power period (min)",
    cats='Condition',
    kind='colored',
    palette=bokeh.palettes.d3['Category20'][5][-1],
    tooltips=[('Position', '@{Cell_position}'),
             ('Period', '@{Max. power period (min)}')],
    p=p
)

p = iqplot.ecdf(
    data=df_total.loc[df_total['Condition'] == '80 nM',:],
    q="Max. power period (min)",
    cats='Condition',
    kind='colored',
    palette=bokeh.palettes.d3['Category20'][5][-3],
    tooltips=[('Position', '@{Cell_position}'),
             ('Period', '@{Max. power period (min)}')],
    p=p
)

bokeh.io.show(p)

C:\Users\pperez\.conda\envs\devbio-napari-cupy\lib\site-packages\bokeh\io\notebook.py:487: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  publish_display_data(data, metadata, source, transient=transient, **kwargs)


In [ ]:
signals_control

In [ ]:
wAn.compute_spectrum?

In [10]:
# Load signals
signals_control = pd.read_csv('./control', delimiter=',', index_col=['Unnamed: 0'])
signals_40nm = pd.read_csv('./40nM', delimiter=',', index_col=['Unnamed: 0'])
signals_80nm = pd.read_csv('./80nM', delimiter=',', index_col=['Unnamed: 0'])
all_signals_norm = normalize_df(pd.concat([signals_control, signals_40nm, signals_80nm], axis=1, ignore_index=False))
all_signals_norm

,ppf005_xy002_0,ppf005_xy003_0,ppf005_xy003_2,ppf005_xy005_0,ppf005_xy007_1,ppf005_xy008_0,ppf005_xy011_0,ppf005_xy013_0,ppf005_xy014_0,ppf005_xy015_0,...,ppf005_xy162_0,ppf005_xy163_0,ppf005_xy164_0,ppf005_xy166_0,ppf005_xy168_0,ppf005_xy170_0,ppf005_xy171_0,ppf005_xy173_0,ppf005_xy173_1,ppf005_xy174_0
0,0.000000,0.000000,0.068732,0.025722,0.005980,0.022744,1.000000,0.619708,0.007485,0.012353,...,0.090868,0.032866,0.016361,0.038333,0.116385,0.051868,0.022978,0.075243,0.127750,0.156060
1,0.098967,0.004073,0.244902,0.000000,0.101524,0.255782,0.767681,0.355789,0.000000,0.172433,...,0.384335,0.033471,0.017004,0.032127,0.413997,0.049447,0.000000,0.046015,0.087751,0.178369
2,0.051715,0.030329,0.282711,0.041501,0.115689,0.324801,0.612197,0.173352,0.148237,0.200117,...,0.460294,0.100393,0.087763,0.109835,0.420411,0.081290,0.131024,0.061800,0.083900,0.342623
3,0.066805,0.026902,0.122940,0.180595,0.027808,0.170014,0.434443,0.371791,0.162800,0.073922,...,0.318436,0.142871,0.141237,0.099626,0.331678,0.060937,0.182034,0.217766,0.213103,0.216321
4,0.070593,0.037035,0.141573,0.258565,0.015926,0.063168,0.517429,0.609833,0.033008,0.137886,...,0.123668,0.120156,0.113428,0.030787,0.141797,0.030558,0.078607,0.184539,0.216006,0.168334
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,0.168083,0.028577,0.036919,0.059261,0.127399,0.050419,0.039874,0.039937,0.016909,0.066495,...,0.266114,0.067587,0.013648,0.037101,0.064931,0.049092,0.063011,0.187297,0.005436,0.053202
76,0.149288,0.019700,0.031981,0.069946,0.129446,0.064932,0.029704,0.044377,0.025985,0.061732,...,0.038888,0.081922,0.005533,0.013242,0.081223,0.055930,0.041343,0.186542,0.020168,0.053284
77,0.125185,0.019569,0.031667,0.067613,0.138113,0.049130,0.035645,0.035008,0.053329,0.066686,...,0.029455,0.064265,0.007915,0.032916,0.073030,0.021135,0.031446,0.182582,0.019829,0.049026
78,0.164832,0.015243,0.042008,0.071755,0.140352,0.042848,0.028647,0.048146,0.030433,0.075857,...,0.018243,0.074630,0.015240,0.024829,0.097637,0.035199,0.027425,0.181067,0.019874,0.035791


In [49]:
signals_40nm_norm = normalize_df(signals_40nm)

In [308]:
# App to record peaks

from bokeh.plotting import figure, curdoc
from bokeh.models import Button, ColumnDataSource, Slider, Select, Span
import bokeh.layouts

# SInitial values
t = np.linspace(0,79,80)
signals_df = all_signals_norm.copy(deep=True)
selected_signal = signals_df.columns.values[0]
signal = signals_df[selected_signal]
# signal=signal/(np.max(signal))
peaks=[]
heights=[]
# Initial parameters for find_peaks
initial_prominence = 0.5
initial_height = 0.01
threshold=np.zeros(len(t))

# Data dataframe to store data
global peak_data
peak_data = pd.DataFrame()

# Find Peaks function
def find_peaks_with_params(signal, prominence, height):
    peaks, properties = scipy.signal.find_peaks(signal, prominence=prominence, height=height)
    return peaks, properties['peak_heights']

# Create ColumnDataSource
source1 = ColumnDataSource(data=dict(t=t, signal=signal))
source2 = ColumnDataSource(data=dict(peaks=peaks, heights=heights))
source3 = ColumnDataSource(data=dict(t=t, threshold=threshold))

# Create Bokeh Figure
plot = figure(title='normalized  '+signals_df.columns.values[0], x_range=(np.min(t), np.max(t)), y_range=(np.min(signal), np.max(signal)+0.01), width=1000, height=800)
plot.circle('t', 'signal', source=source1, line_width=2, line_color='blue', legend_label='Signal')
plot.line('t', 'signal', source=source1, line_width=2, line_color='blue', legend_label='Signal')
plot.circle('peaks', 'heights', source=source2, size=8, color='red', legend_label='Peaks', line_width=2, line_color='black')
plot.line('t', 'threshold', source=source3, line_color='green', line_dash='dashed', legend_label='Height threshold')

# Callback function for filtering signal

# Callback function for dropdown menu
def update_signal(attr, old, new):
    selected_signal = signal_select.value
    signal = signals_df[selected_signal]
    
    # Update data source
    source1.data = dict(t=t, signal=signal)
    
    # Update peaks and heights based on new signal
    prominence_value = prominence_slider.value
    height_value = height_slider.value
    peaks, heights = find_peaks_with_params(signal, prominence_value, height_value)
    source2.data = dict(peaks=peaks, heights=heights)
    
    # Update plot title
    plot.title.text = selected_signal

# Callback function for sliders using on_change
def update_data(attr, old, new):
    # Retrieve values from slider
    prominence_value = prominence_slider.value
    height_value = height_slider.value
    selected_signal = signal_select.value
    signal = signals_df[selected_signal]    
    # Update peaks and heights based on new prominence value
    peaks, heights = find_peaks_with_params(signal, prominence_value, height_value)
    threshold = np.ones(len(t))*height_value
    # Update data source
    source2.data = dict(peaks=peaks, heights=heights)
    source3.data = dict(t=t, threshold=threshold)
    

# Save data function
def save_data():
    selected_signal = signal_select.value
    signal = signals_df[selected_signal]
    prominence_value = prominence_slider.value
    height_value = height_slider.value
    peaks, peak_heights = find_peaks_with_params(signal, prominence_value, height_value)
    data = {
        'Signal': [selected_signal]*len(peaks),
        'Peak pos': peaks,
        'Peak heights': peak_heights,
        'Threshold': [height_value]*len(peaks),
        'Peak proms': [prominence_value]*len(peaks)
    }
    df = pd.DataFrame(data)
    global peak_data
    peak_data=pd.concat([peak_data, df], ignore_index=True)
    
    # Print or save the DataFrame (adjust as needed)
    print(df)
    
    # Move to the next signal in the dropdown menu
    signal_index = list(signals_df.keys()).index(selected_signal)
    try:
        next_index = (signal_index + 1) % len(signals_df)
        next_signal = list(signals_df.keys())[next_index]
        signal_select.value = next_signal
    except IndexError:
        print('{selected_signal} is the last signal of the dataset'.format(selected_signal=selected_signal))
        
# Create sliders with on_change callback
prominence_slider = Slider(title='Prominence', value=initial_prominence, start=0.0, end=1.0, step=0.01)
prominence_slider.on_change('value', update_data)
height_slider = Slider(title='Height', value=initial_height, start=0.01, end=1.0, step=0.01)
height_slider.on_change('value', update_data)

# Create dropdown menu
signal_select = Select(title='Select Signal:', value=selected_signal, options=list(signals_df.keys()))
signal_select.on_change('value', update_signal)

# Create a button to save data
save_button = Button(label="Save Data", button_type="success")
save_button.on_click(save_data)

# Create a checkbox to consider valid the signal - TO DO

slider_layout = bokeh.layouts.column(
    bokeh.layouts.Spacer(height=30),
    prominence_slider,
    bokeh.layouts.Spacer(height=15),
    height_slider,
)

#Dropdown and save button
dropdown_layout = bokeh.layouts.column(
    bokeh.layouts.Spacer(height=30),
    signal_select,
    save_button,
)

# Set up layout
norm_layout = bokeh.layouts.row(
    plot,
    bokeh.layouts.Spacer(width=15),
    slider_layout,
    dropdown_layout
)
# Add layout to the current document
def norm_app(doc):
    doc.add_root(norm_layout)

notebook_url = 'localhost:8888'
bokeh.io.show(norm_app, notebook_url=notebook_url)

INFO:bokeh.server.server:Starting Bokeh server version 2.4.3 (running on Tornado 6.2)
INFO:bokeh.server.tornado:User authentication hooks NOT provided (default user enabled)


INFO:tornado.access:200 GET /autoload.js?bokeh-autoload-element=123475&bokeh-absolute-url=http://localhost:51865&resources=none (::1) 39.52ms
INFO:tornado.access:101 GET /ws (::1) 0.00ms
INFO:bokeh.server.views.ws:WebSocket connection opened
INFO:bokeh.server.views.ws:ServerConnection created


In [ ]:
df_peaks_80nm

INFO:tornado.access:200 GET /autoload.js?bokeh-autoload-element=1215&bokeh-absolute-url=http://localhost:55312&resources=none (::1) 23.00ms
ERROR:bokeh.server.views.ws:Refusing websocket connection from Origin 'http://localhost:8889';                       use --allow-websocket-origin=localhost:8889 or set BOKEH_ALLOW_WS_ORIGIN=localhost:8889 to permit this; currently we allow origins {'localhost:8888'}


In [56]:
df_peaks_control = pd.read_csv('./peak_data_control_40nM', index_col=['Unnamed: 0'])[:173]
df_peaks_control = df_peaks_control.drop(df_peaks_control.loc[df_peaks_control['Signal'] == 'ppf005_xy021_1',:].index)
df_peaks_control = df_peaks_control.drop(df_peaks_control.loc[df_peaks_control['Signal'] == 'ppf005_xy031_0',:].index)

In [57]:
df_peaks_80nm = pd.read_csv('./peak_data_80nM', index_col=['Unnamed: 0'])
df_peaks_80nm = df_peaks_80nm.drop(df_peaks_80nm.loc[df_peaks_80nm['Signal'] == 'ppf005_xy123_0',:].index)

In [58]:
value_count_control = df_peaks_control['Signal'].value_counts()
value_count_80nm = df_peaks_80nm['Signal'].value_counts()

In [48]:
len(all_signals_norm.iloc[:,0])

80

In [173]:
from bokeh.plotting import figure, show, curdoc
from bokeh.models import Button, ColumnDataSource, Slider, Select, Span, CustomJS
from bokeh.io import output_notebook
import numpy as np
from matplotlib import cm
import nd2
import bokeh.layouts


# Load your image stack
image_stack_path = Path('./raw/ppf005_xy002.nd2')
image_stack = nd2.imread(image_stack_path.as_posix())
image_stack = image_stack[:,0,:,:]

time_domain = np.asarray(np.linspace(0,79, 80),dtype=np.uint)
ind_images = [image_stack[i,:,:] for i in time_domain]
data={'img':ind_images[0]}
source=ColumnDataSource(data=data)

# Set up Bokeh

#output_notebook()

# Create a Bokeh figure
p = figure(x_range=(0, image_stack.shape[1]), y_range=(0, image_stack.shape[2]))
color_mapper = cm.get_cmap('gray')
img_renderer = p.image(image='img', source=source)

# Initial time point
initial_time_point = 0

# Display the initial image
color_mapper = cm.get_cmap('gray')

# Remove the axis
p.axis.visible = False
p.grid.visible = False

# Create a Slider widget
slider = Slider(start=0, end=image_stack.shape[0] - 1, value=initial_time_point, step=1, title="Time Point")

# Define a callback to update the image when the slider changes
def callback(attr, old, new):
    time_point = slider.value
    new_image = ind_images[time_point]
    
# Attach the callback to the slider
slider.on_change('value', callback)


slider_layout = bokeh.layouts.column(
    bokeh.layouts.Spacer(height=30),
    slider
)

norm_layout = bokeh.layouts.row(
    p,
    bokeh.layouts.Spacer(width=15),
    slider_layout,
)
def norm_app(doc):
    doc.add_root(norm_layout)

bokeh.io.show(norm_app, notebook_url=notebook_url)

In [174]:
from bokeh.plotting import figure, show, curdoc
from bokeh.models import Button, ColumnDataSource, Slider, Select, Span, CustomJS
from bokeh.io import output_notebook
import numpy as np
from matplotlib import cm
import nd2
import bokeh.layouts


# Load your image stack
image_stack_path = Path('./raw/ppf005_xy002.nd2')
image_stack = nd2.imread(image_stack_path.as_posix())
image_stack = image_stack[:,0,:,:]

time_domain = np.asarray(np.linspace(0,79, 80),dtype=np.uint)
ind_images = [image_stack[i,:,:] for i in time_domain]
data={'img':ind_images[0]}
source=ColumnDataSource(data=data)

# Set up Bokeh

#output_notebook()

# Create a Bokeh figure
p = figure(x_range=(0, image_stack.shape[1]), y_range=(0, image_stack.shape[2]))
color_mapper = cm.get_cmap('gray')
p.image(image='img', source=source)

# Initial time point
initial_time_point = 0

# Display the initial image
color_mapper = cm.get_cmap('gray')

# Remove the axis
p.axis.visible = False
p.grid.visible = False

# Create a Slider widget
slider = Slider(start=0, end=image_stack.shape[0] - 1, value=initial_time_point, step=1, title="Time Point")

# Define a callback to update the image when the slider changes
def callback(attr, old, new):
    time_point = slider.value
    new_image = ind_images[time_point]
    
# Attach the callback to the slider
slider.on_change('value', callback)


slider_layout = bokeh.layouts.column(
    bokeh.layouts.Spacer(height=30),
    slider
)

norm_layout = bokeh.layouts.row(
    p,
    bokeh.layouts.Spacer(width=15),
    slider_layout,
)
def norm_app(doc):
    doc.add_root(norm_layout)

bokeh.io.show(norm_app, notebook_url=notebook_url)

In [ ]:
# This is a implementation of a slider to visualize a time-lapse
from bokeh.plotting import figure, show, curdoc
from bokeh.models import Button, ColumnDataSource, Slider, Select, Span, CustomJS
from bokeh.io import output_notebook
import numpy as np
from matplotlib import cm
import nd2
import bokeh.layouts


# Load your image stack
image_stack_path = Path('./raw/ppf005_xy002.nd2')
image_stack = nd2.imread(image_stack_path.as_posix())
image_stack = image_stack[:,0,:,:]

time_domain = np.asarray(np.linspace(0,79, 80),dtype=np.uint)
ind_images = [image_stack[i,:,:] for i in time_domain]
data={'img':[ind_images[0]]}
source=ColumnDataSource(data=data)

# Set up Bokeh

#output_notebook()

# Create a Bokeh figure
p = figure(x_range=(0, image_stack.shape[1]), y_range=(0, image_stack.shape[2]))
color_mapper = cm.get_cmap('gray')
# p.image(image='img', source=source,x=0, y=0, dw=image_stack.shape[1], dh=image_stack.shape[2])
#p.image(image='img', source=source,x=0, y=0, dw=image_stack.shape[1], dh=image_stack.shape[2])
#p.image(image=[ind_images[0]],x=0, y=0, dw=image_stack.shape[1], dh=image_stack.shape[2])
p.image(image='img', x=0, y=0, dw=image_stack.shape[1], dh=image_stack.shape[2],source=source, palette='Greys256')
# Initial time point
initial_time_point = 0

# Display the initial image
# Remove the axis
p.axis.visible = False
p.grid.visible = False

# Create a Slider widget
slider = Slider(start=0, end=image_stack.shape[0] - 1, value=initial_time_point, step=1, title="Time Point")

# Define a callback to update the image when the slider changes
def callback(attr, old, new):
    time_point = slider.value
    new_image = ind_images[time_point]
    source.data = {'img':[new_image]}

    
# Attach the callback to the slider
slider.on_change('value', callback)


slider_layout = bokeh.layouts.column(
    bokeh.layouts.Spacer(height=30),
    slider
)

norm_layout = bokeh.layouts.row(
    p,
    bokeh.layouts.Spacer(width=15),
    slider_layout,
)
def norm_app(doc):
    doc.add_root(norm_layout)

bokeh.io.show(norm_app, notebook_url=notebook_url)

In [ ]:
# App to record peaks
# This one implements the time of death

from bokeh.plotting import figure, curdoc
from bokeh.models import Button, ColumnDataSource, Slider, Select, Span, CustomJS
import bokeh.layouts
import nd2
from pathlib import Path, WindowsPath
from matplotlib import cm
from skimage.io import imread


# Display BF
display = True

# SInitial values
t = np.linspace(0,79,80)
signals_df = all_signals_norm.copy(deep=True)
selected_signal = signals_df.columns.values[0]
signal = signals_df[selected_signal]

# signal=signal/(np.max(signal))
peaks=[]
heights=[]
# Initial parameters for find_peaks
initial_prominence = 0.5
initial_height = 0.01
threshold=np.zeros(len(t))

# Initial parameters for timepoint slider
initial_tf = 0
v_line = np.linspace(0,1,80)
tf = np.ones(len(v_line))

# Load and display initial image
image_path = Path('./output/'+selected_signal[:-2]+'.tif')
image_stack = imread(image_path.as_posix())[:,:,:]
img_data={'img':[image_stack[initial_tf]], 'img_stack':[image_stack], 'path':[image_path.as_posix()]}

# Data dataframe to store data
global peak_data
peak_data = pd.DataFrame()

# Find Peaks function
def find_peaks_with_params(signal, prominence, height):
    peaks, properties = scipy.signal.find_peaks(signal, prominence=prominence, height=height)
    return peaks, properties['peak_heights']

# Create ColumnDataSource
source1 = ColumnDataSource(data=dict(t=t, signal=signal))
source2 = ColumnDataSource(data=dict(peaks=peaks, heights=heights))
source3 = ColumnDataSource(data=dict(t=t, threshold=threshold))
source4 = ColumnDataSource(data=dict(tf=tf, v_line=v_line))
source5 = ColumnDataSource(data=img_data)

# Create Bokeh figure to display signal and peaks
plot = figure(title='normalized  '+signals_df.columns.values[0], x_range=(np.min(t), np.max(t)), y_range=(np.min(signal), np.max(signal)+0.01), width=1000, height=800)
plot.circle('t', 'signal', source=source1, line_width=2, line_color='blue', legend_label='Signal')
plot.line('t', 'signal', source=source1, line_width=2, line_color='blue', legend_label='Signal')
plot.circle('peaks', 'heights', source=source2, size=8, color='red', legend_label='Peaks', line_width=2, line_color='black')
plot.line('t', 'threshold', source=source3, line_color='green', line_dash='dashed', legend_label='Height threshold')
plot.line('tf', 'v_line', source=source4,line_color='red', line_dash='dashed', legend_label='Time of death')

# Create Bokeh figure to display BF images
if display == True:
    bf_display = figure(x_range=(0, image_stack.shape[1]), y_range=(0, image_stack.shape[2]))
    #bf_display = figure(x_range=(0, image_stack.shape[1]), y_range=(0, image_stack.shape[2]), width=1000, height=800)
    bf_display.image(image='img', x=0, y=0, dw=image_stack.shape[1], dh=image_stack.shape[2],source=source5, palette='Greys256')



# Callback function for filtering signal - not implemented for now

# Callback function for dropdown menu
def update_signal(attr, old, new):
    selected_signal = signal_select.value
    signal = signals_df[selected_signal]
    
    # Update data source
    source1.data = dict(t=t, signal=signal)
    
    # Update peaks and heights based on new signal
    prominence_value = prominence_slider.value
    height_value = height_slider.value
    peaks, heights = find_peaks_with_params(signal, prominence_value, height_value)
    source2.data = dict(peaks=peaks, heights=heights)
    
    # Update plot title
    plot.title.text = selected_signal
    
    # Update BF display
    image_path = Path('./output/'+selected_signal[:-2]+'.tif')
    image_stack = imread(image_path.as_posix())[:,:,:]
    img_data={'img':[image_stack[0]], 'img_stack':[image_stack], 'path':[image_path.as_posix()]}
    source5.data = img_data
    tf_slider.value = 0

# Callback function for sliders using on_change
def update_peaks(attr, old, new):
    # Retrieve values from slider
    prominence_value = prominence_slider.value
    height_value = height_slider.value
    selected_signal = signal_select.value
    signal = signals_df[selected_signal]    
    # Update peaks and heights based on new prominence value
    peaks, heights = find_peaks_with_params(signal, prominence_value, height_value)
    threshold = np.ones(len(t))*height_value
    # Update data source
    source2.data = dict(peaks=peaks, heights=heights)
    source3.data = dict(t=t, threshold=threshold)

# Callback function for vertical slider
def update_tf(attr, old, new):
    
    # Neccessary a check to change the signal only if a different signal in the dropdown menu is selected!!!
   # Update line in signal plot
    index = tf_slider.value
    tf = np.ones(len(v_line))*index
    source4.data = dict(tf=tf, v_line=v_line)
    selected_time_lapse = signal_select.value
    image_path = Path('./output/'+selected_time_lapse[:-2]+'.tif').as_posix()
    if image_path != source5.data['path'][0]:
        image_path = Path('./output/'+selected_image[:-2]+'.tif')
        image_stack = nd2.imread(image_path.as_posix())[:,:,:]
    else:
        image_stack = source5.data['img_stack'][0]
    # Update tf in BF display
    if display == True:
        new_image = image_stack[index]
        source5.data['img'] = [new_image]

# Save data function
def save_data():
    selected_signal = signal_select.value
    signal = signals_df[selected_signal]
    prominence_value = prominence_slider.value
    height_value = height_slider.value
    time_of_death = tf_slider.value
    peaks, peak_heights = find_peaks_with_params(signal, prominence_value, height_value)
    data = {
        'Signal': [selected_signal]*len(peaks),
        'Peak pos': peaks,
        'Peak heights': peak_heights,
        'Threshold': [height_value]*len(peaks),
        'Peak proms': [prominence_value]*len(peaks),
        'Time of death': [time_of_death]*len(peaks)
    }
    df = pd.DataFrame(data)
    global peak_data
    peak_data=pd.concat([peak_data, df], ignore_index=True)
    
    # Print or save the DataFrame (adjust as needed)
    print(df)
    
    # Move to the next signal in the dropdown menu
    signal_index = list(signals_df.keys()).index(selected_signal)
    try:
        next_index = (signal_index + 1) % len(signals_df)
        next_signal = list(signals_df.keys())[next_index]
        signal_select.value = next_signal
    except IndexError:
        print('{selected_signal} is the last signal of the dataset'.format(selected_signal=selected_signal))
        
# Create sliders with on_change callback
prominence_slider = Slider(title='Prominence', value=initial_prominence, start=0.0, end=1.0, step=0.01)
prominence_slider.on_change('value', update_peaks)
height_slider = Slider(title='Height', value=initial_height, start=0.01, end=1.0, step=0.01)
height_slider.on_change('value', update_peaks)
tf_slider = Slider(title='Time of death', value=initial_tf, start=0, end=(len(signal)-1), step=1)
tf_slider.on_change('value', update_tf)

# Create dropdown menu
signal_select = Select(title='Select Signal:', value=selected_signal, options=list(signals_df.keys()))
signal_select.on_change('value', update_signal)

# Create a button to save data
save_button = Button(label="Save Data", button_type="success")
save_button.on_click(save_data)

# Create a checkbox to consider valid the signal - TO DO

slider_layout = bokeh.layouts.column(
    bokeh.layouts.Spacer(height=30),
    prominence_slider,
    bokeh.layouts.Spacer(height=15),
    height_slider,
    tf_slider,
)

#Dropdown and save button
dropdown_layout = bokeh.layouts.column(
    bokeh.layouts.Spacer(height=30),
    signal_select,
    save_button,
)

# Set up layout
if display == True:
    norm_layout = bokeh.layouts.row(
        plot,
        bf_display,
        bokeh.layouts.Spacer(width=15),
        slider_layout,
        dropdown_layout
    )
else:
    norm_layout = bokeh.layouts.row(
        plot,
        #bf_display,
        bokeh.layouts.Spacer(width=15),
        slider_layout,
        dropdown_layout
    )
# Add layout to the current document
def norm_app(doc):
    doc.add_root(norm_layout)

bokeh.io.show(norm_app, notebook_url=notebook_url)

In [ ]:
# App to record peaks
# This one implements the time of death and the possibility to hide the display

from bokeh.plotting import figure, curdoc
from bokeh.models import Button, ColumnDataSource, Slider, Select, Span, CustomJS, CheckboxGroup
import bokeh.layouts
import nd2
from pathlib import Path, WindowsPath
from matplotlib import cm
from skimage.io import imread


# Display BF
global display
display = True
# SInitial values
t = np.linspace(0,79,80)
signals_df = all_signals_norm.copy(deep=True)
selected_signal = signals_df.columns.values[0]
signal = signals_df[selected_signal]

# signal=signal/(np.max(signal))
peaks=[]
heights=[]
# Initial parameters for find_peaks
initial_prominence = 0.5
initial_height = 0.01
threshold=np.zeros(len(t))

# Initial parameters for timepoint slider
initial_tf = 0
v_line = np.linspace(0,1,80)
tf = np.ones(len(v_line))

# Load and display initial image
image_path = Path('./output/'+selected_signal[:-2]+'.tif')
image_stack = imread(image_path.as_posix())[:,:,:]
img_data={'img':[image_stack[initial_tf]], 'img_stack':[image_stack], 'path':[image_path.as_posix()]}

# Data dataframe to store data
global peak_data
peak_data = pd.DataFrame()

# Find Peaks function
def find_peaks_with_params(signal, prominence, height):
    peaks, properties = scipy.signal.find_peaks(signal, prominence=prominence, height=height)
    return peaks, properties['peak_heights']

# Create ColumnDataSource
source1 = ColumnDataSource(data=dict(t=t, signal=signal))                    # Signal and time domain
source2 = ColumnDataSource(data=dict(peaks=peaks, heights=heights))          # Peaks and peak heights
source3 = ColumnDataSource(data=dict(t=t, threshold=threshold))              # Time domain and height threshold
source4 = ColumnDataSource(data=dict(tf=tf, v_line=v_line))                  # Time frame and vertical line
source5 = ColumnDataSource(data=img_data)                                    # Image data for BF display

# Create Bokeh figure to display signal and peaks
plot = figure(title='normalized  '+signals_df.columns.values[0], x_range=(np.min(t), np.max(t)), y_range=(np.min(signal), np.max(signal)+0.01), width=1000, height=800)
plot.circle('t', 'signal', source=source1, line_width=2, line_color='blue', legend_label='Signal')
plot.line('t', 'signal', source=source1, line_width=2, line_color='blue', legend_label='Signal')
plot.circle('peaks', 'heights', source=source2, size=8, color='red', legend_label='Peaks', line_width=2, line_color='black')
plot.line('t', 'threshold', source=source3, line_color='green', line_dash='dashed', legend_label='Height threshold')
plot.line('tf', 'v_line', source=source4,line_color='red', line_dash='dashed', legend_label='Time of death')

# Create Bokeh figure to display BF images
if display == True:
    bf_display = figure(x_range=(0, image_stack.shape[1]), y_range=(0, image_stack.shape[2]))
    #bf_display = figure(x_range=(0, image_stack.shape[1]), y_range=(0, image_stack.shape[2]), width=1000, height=800)
    bf_display.image(image='img', x=0, y=0, dw=image_stack.shape[1], dh=image_stack.shape[2],source=source5, palette='Greys256')



# Callback function for filtering signal - not implemented for now

# Callback function for dropdown menu
def update_signal(attr, old, new):
    selected_signal = signal_select.value
    signal = signals_df[selected_signal]
    
    # Update data source
    source1.data = dict(t=t, signal=signal)
    
    # Update peaks and heights based on new signal
    prominence_value = prominence_slider.value
    height_value = height_slider.value
    peaks, heights = find_peaks_with_params(signal, prominence_value, height_value)
    source2.data = dict(peaks=peaks, heights=heights)
    
    # Update plot title
    plot.title.text = selected_signal
    
    # Update BF display
    image_path = Path('./output/'+selected_signal[:-2]+'.tif')
    image_stack = imread(image_path.as_posix())[:,:,:]
    img_data={'img':[image_stack[0]], 'img_stack':[image_stack], 'path':[image_path.as_posix()]}
    source5.data = img_data
    tf_slider.value = 0

# Callback function for sliders using on_change
def update_peaks(attr, old, new):
    # Retrieve values from slider
    prominence_value = prominence_slider.value
    height_value = height_slider.value
    selected_signal = signal_select.value
    signal = signals_df[selected_signal]    
    # Update peaks and heights based on new prominence value
    peaks, heights = find_peaks_with_params(signal, prominence_value, height_value)
    threshold = np.ones(len(t))*height_value
    # Update data source
    source2.data = dict(peaks=peaks, heights=heights)
    source3.data = dict(t=t, threshold=threshold)

# Callback function for vertical slider
def update_tf(attr, old, new):
    
    # Neccessary a check to change the signal only if a different signal in the dropdown menu is selected!!!
   # Update line in signal plot
    index = tf_slider.value
    tf = np.ones(len(v_line))*index
    source4.data = dict(tf=tf, v_line=v_line)
    selected_time_lapse = signal_select.value
    image_path = Path('./output/'+selected_time_lapse[:-2]+'.tif').as_posix()
    if image_path != source5.data['path'][0]:
        image_path = Path('./output/'+selected_image[:-2]+'.tif')
        image_stack = nd2.imread(image_path.as_posix())[:,:,:]
    else:
        image_stack = source5.data['img_stack'][0]
    # Update tf in BF display
    if display == True:
        new_image = image_stack[index]
        source5.data['img'] = [new_image]

# Save data function
def save_data():
    selected_signal = signal_select.value
    signal = signals_df[selected_signal]
    prominence_value = prominence_slider.value
    height_value = height_slider.value
    time_of_death = tf_slider.value
    peaks, peak_heights = find_peaks_with_params(signal, prominence_value, height_value)
    data = {
        'Signal': [selected_signal]*len(peaks),
        'Peak pos': peaks,
        'Peak heights': peak_heights,
        'Threshold': [height_value]*len(peaks),
        'Peak proms': [prominence_value]*len(peaks),
        'Time of death': [time_of_death]*len(peaks)
    }
    df = pd.DataFrame(data)
    global peak_data
    peak_data=pd.concat([peak_data, df], ignore_index=True)
    
    # Print or save the DataFrame (adjust as needed)
    print(df)
    
    # Move to the next signal in the dropdown menu
    signal_index = list(signals_df.keys()).index(selected_signal)
    try:
        next_index = (signal_index + 1) % len(signals_df)
        next_signal = list(signals_df.keys())[next_index]
        signal_select.value = next_signal
    except IndexError:
        print('{selected_signal} is the last signal of the dataset'.format(selected_signal=selected_signal))

# Callback function to toggle BF display
def callback_display(attr, new, old):
    global display 
    display = not display
    bf_display.visible = display

        
# Create sliders with on_change callback
prominence_slider = Slider(title='Prominence', value=initial_prominence, start=0.0, end=1.0, step=0.01)
prominence_slider.on_change('value', update_peaks)
height_slider = Slider(title='Height', value=initial_height, start=0.01, end=1.0, step=0.01)
height_slider.on_change('value', update_peaks)
tf_slider = Slider(title='Time of death', value=initial_tf, start=0, end=(len(signal)-1), step=1)
tf_slider.on_change('value', update_tf)

# Create dropdown menu
signal_select = Select(title='Select Signal:', value=selected_signal, options=list(signals_df.keys()))
signal_select.on_change('value', update_signal)

# Create a button to save data
save_button = Button(label="Save Data", button_type="success")
save_button.on_click(save_data)

# Create a checkbox to display BF
checkbox_display = CheckboxGroup(labels=['Display BF'], active=[0,1])
checkbox_display.on_change('active', callback_display)

# Create a checkbox to consider valid the signal - TO DO



slider_layout = bokeh.layouts.column(
    bokeh.layouts.Spacer(height=30),
    prominence_slider,
    bokeh.layouts.Spacer(height=15),
    height_slider,
    tf_slider,
)

#Dropdown and save button
dropdown_layout = bokeh.layouts.column(
    bokeh.layouts.Spacer(height=30),
    signal_select,
    save_button,
    checkbox_display
)

# Set up layout
norm_layout = bokeh.layouts.row(
    plot,
    bf_display,
    bokeh.layouts.Spacer(width=15),
    slider_layout,
    dropdown_layout,
)

# Add layout to the current document
def norm_app(doc):
    doc.add_root(norm_layout)

bokeh.io.show(norm_app, notebook_url=notebook_url)

In [27]:
# App to record peaks
# This one implements the time of death and the possibility to hide the display
# Implement manual selection of peaks

from bokeh.plotting import figure, curdoc
from bokeh.models import Button, ColumnDataSource, Slider, Select, Span, CustomJS, CheckboxGroup, Arrow, NormalHead
import bokeh.layouts
import nd2
from pathlib import Path, WindowsPath
from matplotlib import cm
from skimage.io import imread


# Display BF
global display
display = True
# SInitial values
t = np.linspace(0,79,80)
signals_df = all_signals_norm.copy(deep=True)
selected_signal = signals_df.columns.values[0]
signal = signals_df[selected_signal]

# signal=signal/(np.max(signal))
peaks=[]
peak_heights=[]
# Initial parameters for find_peaks
initial_prominence = 0.5
initial_height = 0.01
threshold=np.zeros(len(t))

# Initial parameters for timepoint slider
initial_tf = 0
v_line = np.linspace(0,1,80)
tf = np.ones(len(v_line))

# Load and display initial image
image_path = Path('./output/'+selected_signal[:-2]+'.tif')
image_stack = imread(image_path.as_posix())[:,:,:]
img_data={'img':[image_stack[initial_tf]], 'img_stack':[image_stack], 'path':[image_path.as_posix()]}

# Data dataframe to store data
global peak_data
peak_data = pd.DataFrame()

# Find Peaks function
def find_peaks_with_params(signal, prominence, height):
    peaks, properties = scipy.signal.find_peaks(signal, prominence=prominence, height=height)
    return peaks, properties['peak_heights']

# Create ColumnDataSource
source1 = ColumnDataSource(data=dict(t=t, signal=signal))                    # Signal and time domain
source2 = ColumnDataSource(data=dict(peaks=peaks, heights=peak_heights))          # Peaks and peak heights
source3 = ColumnDataSource(data=dict(t=t, threshold=threshold))              # Time domain and height threshold
source4 = ColumnDataSource(data=dict(tf=tf, v_line=v_line))                  # Time frame and vertical line
source5 = ColumnDataSource(data=img_data)                                    # Image data for BF display

# Create Bokeh figure to display signal and peaks
plot = figure(title='normalized  '+signals_df.columns.values[0], x_range=(np.min(t), np.max(t)), y_range=(np.min(signal), np.max(signal)+0.01), width=1000, height=800, tools=[TapTool()])
plot.circle('t', 'signal', source=source1, line_width=2, line_color='blue', legend_label='Signal')
plot.line('t', 'signal', source=source1, line_width=2, line_color='blue', legend_label='Signal')
plot.circle('peaks', 'heights', source=source2, size=8, color='red', legend_label='Peaks', line_width=2, line_color='black')
plot.line('t', 'threshold', source=source3, line_color='green', line_dash='dashed', legend_label='Height threshold')
plot.line('tf', 'v_line', source=source4,line_color='red', line_dash='dashed', legend_label='Time of death')

# Create Bokeh figure to display BF images
if display == True:
    bf_display = figure(x_range=(0, image_stack.shape[1]), y_range=(0, image_stack.shape[2]))
    #bf_display = figure(x_range=(0, image_stack.shape[1]), y_range=(0, image_stack.shape[2]), width=1000, height=800)
    bf_display.image(image='img', x=0, y=0, dw=image_stack.shape[1], dh=image_stack.shape[2],source=source5, palette='Greys256')



# Callback function for filtering signal - not implemented for now

# Callback function for dropdown menu
def update_signal(attr, old, new):
    selected_signal = signal_select.value
    signal = signals_df[selected_signal]
    
    # Update data source
    source1.data = dict(t=t, signal=signal)
    
    # Update peaks and heights based on new signal
    prominence_value = prominence_slider.value
    height_value = height_slider.value
    peaks, peak_heights = find_peaks_with_params(signal, prominence_value, height_value)
    source2.data = dict(peaks=peaks, heights=peak_heights)
    
    # Update plot title
    plot.title.text = selected_signal
    
    # Update BF display
    image_path = Path('./output/'+selected_signal[:-2]+'.tif')
    image_stack = imread(image_path.as_posix())[:,:,:]
    img_data={'img':[image_stack[0]], 'img_stack':[image_stack], 'path':[image_path.as_posix()]}
    source5.data = img_data
    tf_slider.value = 0

# Callback function for sliders using on_change
def update_peaks(attr, old, new):
    # Retrieve values from slider
    prominence_value = prominence_slider.value
    height_value = height_slider.value
    selected_signal = signal_select.value
    signal = signals_df[selected_signal]    
    # Update peaks and heights based on new prominence value
    peaks, peak_heights = find_peaks_with_params(signal, prominence_value, height_value)
    threshold = np.ones(len(t))*height_value
    # Update data source
    source2.data = dict(peaks=peaks, heights=peak_heights)
    source3.data = dict(t=t, threshold=threshold)

# Callback function for vertical slider
def update_tf(attr, old, new):
    
    # Neccessary a check to change the signal only if a different signal in the dropdown menu is selected!!!
   # Update line in signal plot
    index = tf_slider.value
    tf = np.ones(len(v_line))*index
    source4.data = dict(tf=tf, v_line=v_line)
    selected_time_lapse = signal_select.value
    image_path = Path('./output/'+selected_time_lapse[:-2]+'.tif').as_posix()
    if image_path != source5.data['path'][0]:
        image_path = Path('./output/'+selected_image[:-2]+'.tif')
        image_stack = nd2.imread(image_path.as_posix())[:,:,:]
    else:
        image_stack = source5.data['img_stack'][0]
    # Update tf in BF display
    if display == True:
        new_image = image_stack[index]
        source5.data['img'] = [new_image]

# Save data function
def save_data():
    selected_signal = signal_select.value
    signal = signals_df[selected_signal]
    prominence_value = prominence_slider.value
    height_value = height_slider.value
    time_of_death = tf_slider.value
    peaks, peak_heights = find_peaks_with_params(signal, prominence_value, height_value)
    data = {
        'Signal': [selected_signal]*len(peaks),
        'Peak pos': peaks,
        'Peak heights': peak_heights,
        'Threshold': [height_value]*len(peaks),
        'Peak proms': [prominence_value]*len(peaks),
        'Time of death': [time_of_death]*len(peaks)
    }
    df = pd.DataFrame(data)
    global peak_data
    peak_data=pd.concat([peak_data, df], ignore_index=True)
    
    # Print or save the DataFrame (adjust as needed)
    print(df)
    
    # Move to the next signal in the dropdown menu
    signal_index = list(signals_df.keys()).index(selected_signal)
    try:
        next_index = (signal_index + 1) % len(signals_df)
        next_signal = list(signals_df.keys())[next_index]
        signal_select.value = next_signal
    except IndexError:
        print('{selected_signal} is the last signal of the dataset'.format(selected_signal=selected_signal))

# Callback function to toggle BF display
def callback_display(attr, new, old):
    global display 
    display = not display
    bf_display.visible = display

    
def tap_point(attr, old, new):
    try:
        
        # peaks = source2.data['peaks']
        # peak_heights = source2.data['heights']
        selected_index = source1.selected.indices[0]
        selected_peak = source1.data['t'][selected_index]
        selected_height = source1.data['signal'][selected_index]
        # add peak if peak is not in the previous peak list
        if selected_peak not in peaks:
            new_peaks = {'peaks': np.append(source2.data['peaks'], selected_peak), 'heights': np.append(source2.data['heights'], selected_height)}
        
        elif selected_index in list(map(int, peaks)):
            global position_selected_height
            position_selected_height = np.where(source2.data['heights'] == selected_height)[0][0]
            new_peaks = {'peaks': np.delete(source2.data['peaks'], selected_peak), 'heights': np.delete(source2.data['heights'], position_selected_height)}
        source2.data = new_peaks
        
    except IndexError:
        pass
        
# Create sliders with on_change callback
prominence_slider = Slider(title='Prominence', value=initial_prominence, start=0.0, end=1.0, step=0.01)
prominence_slider.on_change('value', update_peaks)
height_slider = Slider(title='Height', value=initial_height, start=0.01, end=1.0, step=0.01)
height_slider.on_change('value', update_peaks)
tf_slider = Slider(title='Time of death', value=initial_tf, start=0, end=(len(signal)-1), step=1)
tf_slider.on_change('value', update_tf)

# Create dropdown menu
signal_select = Select(title='Select Signal:', value=selected_signal, options=list(signals_df.keys()))
signal_select.on_change('value', update_signal)

# Create a button to save data
save_button = Button(label="Save Data", button_type="success")
save_button.on_click(save_data)

# Create a checkbox to display BF
checkbox_display = CheckboxGroup(labels=['Display BF'], active=[0,1])
checkbox_display.on_change('active', callback_display)

# Add or remove peaks by tapping
# source1.selected.on_change('indices', tap_point)


# Create a checkbox to consider valid the signal - TO DO



slider_layout = bokeh.layouts.column(
    bokeh.layouts.Spacer(height=30),
    prominence_slider,
    bokeh.layouts.Spacer(height=15),
    height_slider,
    tf_slider,
)

#Dropdown and save button
dropdown_layout = bokeh.layouts.column(
    bokeh.layouts.Spacer(height=30),
    signal_select,
    save_button,
    checkbox_display
)

# Set up layout
norm_layout = bokeh.layouts.row(
    plot,
    bf_display,
    bokeh.layouts.Spacer(width=15),
    slider_layout,
    dropdown_layout,
)

# Add layout to the current document
def norm_app(doc):
    doc.add_root(norm_layout)

bokeh.io.show(norm_app, notebook_url=notebook_url)

INFO:bokeh.server.server:Starting Bokeh server version 2.4.3 (running on Tornado 6.4)
INFO:bokeh.server.tornado:User authentication hooks NOT provided (default user enabled)


INFO:tornado.access:200 GET /autoload.js?bokeh-autoload-element=9228&bokeh-absolute-url=http://localhost:52917&resources=none (::1) 65.99ms
INFO:tornado.access:101 GET /ws (::1) 1.00ms
INFO:bokeh.server.views.ws:WebSocket connection opened
INFO:bokeh.server.views.ws:ServerConnection created


In [722]:
# App to record peaks
# This one implements the time of death and the possibility to hide the display
# Implement manual addition of peaks - removal still to be added

from bokeh.plotting import figure, curdoc
from bokeh.models import Button, ColumnDataSource, Slider, Select, Span, CustomJS, CheckboxGroup
import bokeh.layouts
import nd2
from pathlib import Path, WindowsPath
from matplotlib import cm
from skimage.io import imread


# Display BF
global display
display = True
# SInitial values
t = np.linspace(0,79,80)
signals_df = all_signals_norm.copy(deep=True)
selected_signal = signals_df.columns.values[0]
signal = signals_df[selected_signal]

# signal=signal/(np.max(signal))
peaks=[]
peak_heights=[]
# Initial parameters for find_peaks
initial_prominence = 0.5
initial_height = 0.01
threshold=np.zeros(len(t))

# Initial parameters for timepoint slider
initial_tf = 0
v_line = np.linspace(0,1,80)
tf = np.ones(len(v_line))

# Load and display initial image
image_path = Path('./output/'+selected_signal[:-2]+'.tif')
image_stack = imread(image_path.as_posix())[:,:,:]
img_data={'img':[image_stack[initial_tf]], 'img_stack':[image_stack], 'path':[image_path.as_posix()]}

# Data dataframe to store data
global peak_data
peak_data = pd.DataFrame()

# Find Peaks function
def find_peaks_with_params(signal, prominence, height):
    peaks, properties = scipy.signal.find_peaks(signal, prominence=prominence, height=height)
    return peaks, properties['peak_heights']

# Create ColumnDataSource
source1 = ColumnDataSource(data=dict(t=t, signal=signal))                    # Signal and time domain
source2 = ColumnDataSource(data=dict(peaks=peaks, heights=peak_heights))          # Peaks and peak heights
source3 = ColumnDataSource(data=dict(t=t, threshold=threshold))              # Time domain and height threshold
source4 = ColumnDataSource(data=dict(tf=tf, v_line=v_line))                  # Time frame and vertical line
source5 = ColumnDataSource(data=img_data)                                    # Image data for BF display

# Create Bokeh figure to display signal and peaks
plot = figure(title='normalized  '+signals_df.columns.values[0], x_range=(np.min(t), np.max(t)), y_range=(np.min(signal), np.max(signal)+0.01), width=1000, height=800, tools=[TapTool(), BoxZoomTool(), ResetTool()])
plot.circle('t', 'signal', source=source1, line_width=2, line_color='blue', legend_label='Signal', nonselection_alpha=1.0)
plot.line('t', 'signal', source=source1, line_width=2, line_color='blue', legend_label='Signal')
plot.circle('peaks', 'heights', source=source2, size=8, color='red', legend_label='Peaks', line_width=2, line_color='black', nonselection_alpha=1.0)
plot.line('t', 'threshold', source=source3, line_color='green', line_dash='dashed', legend_label='Height threshold')
plot.line('tf', 'v_line', source=source4,line_color='red', line_dash='dashed', legend_label='Time of death')

# Create Bokeh figure to display BF images
if display == True:
    bf_display = figure(x_range=(0, image_stack.shape[1]), y_range=(0, image_stack.shape[2]))
    #bf_display = figure(x_range=(0, image_stack.shape[1]), y_range=(0, image_stack.shape[2]), width=1000, height=800)
    bf_display.image(image='img', x=0, y=0, dw=image_stack.shape[1], dh=image_stack.shape[2],source=source5, palette='Greys256')



# Callback function for filtering signal - not implemented for now

# Callback function for dropdown menu
def update_signal(attr, old, new):
    selected_signal = signal_select.value
    signal = signals_df[selected_signal]
    
    # Update data source
    source1.data = dict(t=t, signal=signal)
    
    # Update peaks and heights based on new signal
    prominence_value = prominence_slider.value
    height_value = height_slider.value
    peaks, peak_heights = find_peaks_with_params(signal, prominence_value, height_value)
    source2.data = dict(peaks=peaks, heights=peak_heights)
    
    # Update plot title
    plot.title.text = selected_signal
    
    # Update BF display
    image_path = Path('./output/'+selected_signal[:-2]+'.tif')
    image_stack = imread(image_path.as_posix())[:,:,:]
    img_data={'img':[image_stack[0]], 'img_stack':[image_stack], 'path':[image_path.as_posix()]}
    source5.data = img_data
    tf_slider.value = 0

# Callback function for sliders using on_change
def update_peaks(attr, old, new):
    # Retrieve values from slider
    prominence_value = prominence_slider.value
    height_value = height_slider.value
    selected_signal = signal_select.value
    signal = signals_df[selected_signal]    
    # Update peaks and heights based on new prominence value
    peaks, peak_heights = find_peaks_with_params(signal, prominence_value, height_value)
    threshold = np.ones(len(t))*height_value
    # Update data source
    source2.data = dict(peaks=peaks, heights=peak_heights)
    source3.data = dict(t=t, threshold=threshold)

# Callback function for vertical slider
def update_tf(attr, old, new):
    
    # Neccessary a check to change the signal only if a different signal in the dropdown menu is selected!!!
   # Update line in signal plot
    index = tf_slider.value
    tf = np.ones(len(v_line))*index
    source4.data = dict(tf=tf, v_line=v_line)
    selected_time_lapse = signal_select.value
    image_path = Path('./output/'+selected_time_lapse[:-2]+'.tif').as_posix()
    if image_path != source5.data['path'][0]:
        image_path = Path('./output/'+selected_image[:-2]+'.tif')
        image_stack = nd2.imread(image_path.as_posix())[:,:,:]
    else:
        image_stack = source5.data['img_stack'][0]
    # Update tf in BF display
    if display == True:
        new_image = image_stack[index]
        source5.data['img'] = [new_image]

# Save data function
def save_data():
    selected_signal = signal_select.value
    signal = signals_df[selected_signal]
    prominence_value = prominence_slider.value
    height_value = height_slider.value
    time_of_death = tf_slider.value
    peaks, peak_heights = find_peaks_with_params(signal, prominence_value, height_value)
    data = {
        'Signal': [selected_signal]*len(peaks),
        'Peak pos': peaks,
        'Peak heights': peak_heights,
        'Threshold': [height_value]*len(peaks),
        'Peak proms': [prominence_value]*len(peaks),
        'Time of death': [time_of_death]*len(peaks)
    }
    df = pd.DataFrame(data)
    global peak_data
    peak_data=pd.concat([peak_data, df], ignore_index=True)
    
    # Print or save the DataFrame (adjust as needed)
    print(df)
    
    # Move to the next signal in the dropdown menu
    signal_index = list(signals_df.keys()).index(selected_signal)
    try:
        next_index = (signal_index + 1) % len(signals_df)
        next_signal = list(signals_df.keys())[next_index]
        signal_select.value = next_signal
    except IndexError:
        print('{selected_signal} is the last signal of the dataset'.format(selected_signal=selected_signal))
    
# Callback function to toggle BF display
def callback_display(attr, new, old):
    global display 
    display = not display
    bf_display.visible = display

    
def tap_point(attr, old, new):
    try:
        
        # peaks = source2.data['peaks']
        # peak_heights = source2.data['heights']
        selected_index = source1.selected.indices[0]
        selected_peak = source1.data['t'][selected_index]
        selected_height = source1.data['signal'][selected_index]
        # add peak if peak is not in the previous peak list
        if selected_peak not in peaks:
            new_peaks = {'peaks': np.append(source2.data['peaks'], selected_peak), 'heights': np.append(source2.data['heights'], selected_height)}
            source2.data = new_peaks
        elif selected_index in peaks:
            global position_selected_height
            global position_selected_peak
            position_selected_height = np.where(source2.data['heights'] == selected_height)[0][0]
            position_selected_peak = np.where(source2.data['peaks'] == selected_peak)[0][0]
            new_peaks = {'peaks': np.delete(source2.data['peaks'], position_selected_peak), 'heights': np.delete(source2.data['heights'], position_selected_height)}
            source2.data = new_peaks
        
    except IndexError:
        pass
        
# Create sliders with on_change callback
prominence_slider = Slider(title='Prominence', value=initial_prominence, start=0.0, end=1.0, step=0.01)
prominence_slider.on_change('value', update_peaks)
height_slider = Slider(title='Height', value=initial_height, start=0.01, end=1.0, step=0.01)
height_slider.on_change('value', update_peaks)
tf_slider = Slider(title='Time of death', value=initial_tf, start=0, end=(len(signal)-1), step=1)
tf_slider.on_change('value', update_tf)

# Create dropdown menu
signal_select = Select(title='Select Signal:', value=selected_signal, options=list(signals_df.keys()))
signal_select.on_change('value', update_signal)

# Create a button to save data
save_button = Button(label="Save Data", button_type="success")
save_button.on_click(save_data)

# Create a checkbox to display BF
checkbox_display = CheckboxGroup(labels=['Display BF'], active=[0,1])
checkbox_display.on_change('active', callback_display)

# Add or remove peaks by tapping
source1.selected.on_change('indices', tap_point)


# Create a checkbox to consider valid the signal - TO DO



slider_layout = bokeh.layouts.column(
    bokeh.layouts.Spacer(height=30),
    prominence_slider,
    bokeh.layouts.Spacer(height=15),
    height_slider,
    tf_slider,
)

#Dropdown and save button
dropdown_layout = bokeh.layouts.column(
    bokeh.layouts.Spacer(height=30),
    signal_select,
    save_button,
    checkbox_display
)

# Set up layout
norm_layout = bokeh.layouts.row(
    plot,
    bf_display,
    bokeh.layouts.Spacer(width=15),
    slider_layout,
    dropdown_layout,
)

# Add layout to the current document
def norm_app(doc):
    doc.add_root(norm_layout)

bokeh.io.show(norm_app, notebook_url=notebook_url)

INFO:bokeh.server.server:Starting Bokeh server version 2.4.3 (running on Tornado 6.2)
INFO:bokeh.server.tornado:User authentication hooks NOT provided (default user enabled)


INFO:tornado.access:200 GET /autoload.js?bokeh-autoload-element=267181&bokeh-absolute-url=http://localhost:55258&resources=none (::1) 68.07ms


In [499]:
position_selected_height

In [15]:
# App to record peaks
# This one implements the time of death and the possibility to hide the display
# Implement manual addition of peaks - removal still to be added
# Implements contour plot

from bokeh.plotting import figure, curdoc, show
from bokeh.models import Button, ColumnDataSource, Slider, Select, Span, CustomJS, CheckboxGroup, ResetTool
from bokeh.models import FactorRange, HoverTool, TapTool, Select, BoxZoomTool

import bokeh.layouts
import nd2
from pathlib import Path, WindowsPath
from skimage.io import imread
import ast

import numpy as np
import pandas as pd

from bokeh.models import FactorRange, Button, ColumnDataSource, HoverTool, TapTool, Slider, Select, Span, BoxZoomTool

import bokeh.io

import iqplot

bokeh.io.output_notebook()


# Turn off warnings
from __future__ import annotations

import logging # isort:skip
log = logging.getLogger(__name__)

from bokeh.util.warnings import BokehUserWarning 
import warnings 
warnings.simplefilter(action='ignore', category=BokehUserWarning)

# Display BF
global display
display = True

global contour
contour = True

# SInitial values
t = np.linspace(0,79,80)
signals_df = all_signals_norm.copy(deep=True)
selected_signal = signals_df.columns.values[0]
signal = signals_df[selected_signal]
contours_path = Path('./')
# signal=signal/(np.max(signal))
peaks=[]
peak_heights=[]
# Initial parameters for find_peaks
initial_prominence = 0.5
initial_height = 0.01
threshold=np.zeros(len(t))

# Initial parameters for timepoint slider
initial_tf = 0
v_line = np.linspace(0,1,80)
tf = np.ones(len(v_line))

# Load and display initial image
image_path = Path('./output/'+selected_signal[:-2]+'.tif')
image_stack = imread(image_path.as_posix())[:,:,:]
img_data={'img':[image_stack[initial_tf]], 'img_stack':[image_stack], 'path':[image_path.as_posix()]}

# Load contour data and plot first one
contour_data = pd.read_csv(contours_path, index_col='Time Frame')
contour_dict = {
                # 'x_coords':ast.literal_eval(contour_data.loc[(contour_data['Cell Name']==selected_signal) & (contour_data['Time Frame'] == initial_tf), 'x_coords'][initial_tf]),
                # 'y_coords':ast.literal_eval(contour_data.loc[(contour_data['Cell Name']==selected_signal) & (contour_data['Time Frame'] == initial_tf), 'y_coords'][initial_tf]),}
                'x_coords':ast.literal_eval(contour_data.loc[(contour_data['Cell Name']==selected_signal), 'x_coords'][initial_tf]),
                'y_coords':ast.literal_eval(contour_data.loc[(contour_data['Cell Name']==selected_signal), 'y_coords'][initial_tf]),}
#                 # 'y_coords':ast.literal_eval(contour_data.loc[contour_data['Cell Name']==selected_signal, 'y_coords'][initial_tf])}
#                 'x_coords':ast.literal_eval(contour_data.loc[contour_data['Cell Name']==selected_signal, 'y_coords'][initial_tf]),
#                 'y_coords':ast.literal_eval(contour_data.loc[contour_data['Cell Name']==selected_signal, 'y_coords'][initial_tf])}


# Data dataframe to store data
global peak_data
peak_data = pd.DataFrame()

# Find Peaks function
def find_peaks_with_params(signal, prominence, height):
    peaks, properties = scipy.signal.find_peaks(signal, prominence=prominence, height=height)
    return peaks, properties['peak_heights']

# Create ColumnDataSource
source1 = ColumnDataSource(data=dict(t=t, signal=signal))                    # Signal and time domain
source2 = ColumnDataSource(data=dict(peaks=peaks, heights=peak_heights))          # Peaks and peak heights
source3 = ColumnDataSource(data=dict(t=t, threshold=threshold))              # Time domain and height threshold
source4 = ColumnDataSource(data=dict(tf=tf, v_line=v_line))                  # Time frame and vertical line
source5 = ColumnDataSource(data=img_data)                                    # Image data for BF display
source6 = ColumnDataSource(data=contour_dict)                                # Data for contour

# Create Bokeh figure to display signal and peaks
plot = figure(title='normalized  '+signals_df.columns.values[0], x_range=(np.min(t), np.max(t)), y_range=(np.min(signal), np.max(signal)+0.01), width=1000, height=800, tools=[TapTool(), BoxZoomTool(), ResetTool()])
plot.circle('t', 'signal', source=source1, line_width=2, line_color='blue', legend_label='Signal', nonselection_alpha=1.0)
plot.line('t', 'signal', source=source1, line_width=2, line_color='blue', legend_label='Signal')
plot.circle('peaks', 'heights', source=source2, size=8, color='red', legend_label='Peaks', line_width=2, line_color='black', nonselection_alpha=1.0)
plot.line('t', 'threshold', source=source3, line_color='green', line_dash='dashed', legend_label='Height threshold')
plot.line('tf', 'v_line', source=source4,line_color='red', line_dash='dashed', legend_label='Time of death')

# Create Bokeh figure to display BF images
if display == True:
    bf_display = figure(x_range=(0, image_stack.shape[1]), y_range=(0, image_stack.shape[2]))
    #bf_display = figure(x_range=(0, image_stack.shape[1]), y_range=(0, image_stack.shape[2]), width=1000, height=800)
    bf_display.image(image='img', x=0, y=0, dw=image_stack.shape[1], dh=image_stack.shape[2],source=source5, palette='Greys256')
    
    if contour == True:
        contour_renderer = bf_display.line('x_coords', 'y_coords', source=source6,line_color='orange', line_dash='solid', legend_label='Contour')



# Callback function for filtering signal - not implemented for now

# Callback function for dropdown menu
def update_signal(attr, old, new):
    selected_signal = signal_select.value
    signal = signals_df[selected_signal]
    
    # Update data source
    source1.data = dict(t=t, signal=signal)
    
    # Update peaks and heights based on new signal
    prominence_value = prominence_slider.value
    height_value = height_slider.value
    peaks, peak_heights = find_peaks_with_params(signal, prominence_value, height_value)
    source2.data = dict(peaks=peaks, heights=peak_heights)
    
    # Update plot title
    plot.title.text = selected_signal
    
    # Update BF display
    image_path = Path('./output/'+selected_signal[:-2]+'.tif')
    image_stack = imread(image_path.as_posix())[:,:,:]
    img_data={'img':[image_stack[0]], 'img_stack':[image_stack], 'path':[image_path.as_posix()]}
    source5.data = img_data
    tf_slider.value = 0
    
    # Update contour
    source6.data['x_coords'] = ast.literal_eval((contour_data.loc[(contour_data['Cell Name']==signal_select.value), 'x_coords'])[0])
    source6.data['y_coords'] = ast.literal_eval((contour_data.loc[(contour_data['Cell Name']==signal_select.value), 'y_coords'])[0])
    
# Callback function for sliders using on_change
def update_peaks(attr, old, new):
    # Retrieve values from slider
    prominence_value = prominence_slider.value
    height_value = height_slider.value
    selected_signal = signal_select.value
    signal = signals_df[selected_signal]    
    # Update peaks and heights based on new prominence value
    peaks, peak_heights = find_peaks_with_params(signal, prominence_value, height_value)
    threshold = np.ones(len(t))*height_value
    # Update data source
    source2.data = dict(peaks=peaks, heights=peak_heights)
    source3.data = dict(t=t, threshold=threshold)

# Callback function for vertical slider
def update_tf(attr, old, new):
    
    # Neccessary a check to change the signal only if a different signal in the dropdown menu is selected!!!
   # Update line in signal plot
    index = tf_slider.value
    tf = np.ones(len(v_line))*index
    source4.data = dict(tf=tf, v_line=v_line)
    selected_time_lapse = signal_select.value
    image_path = Path('./output/'+selected_time_lapse[:-2]+'.tif').as_posix()
    if image_path != source5.data['path'][0]:
        image_path = Path('./output/'+selected_image[:-2]+'.tif')
        image_stack = nd2.imread(image_path.as_posix())[:,:,:]
    else:
        image_stack = source5.data['img_stack'][0]
    # Update tf in BF display
    if display == True:
        new_image = image_stack[index]
        source5.data['img'] = [new_image]
        # source6.data['x_coords'] = ast.literal_eval((contour_data.loc[(contour_data['Cell Name']==signal_select.value) & (contour_data['Time Frame'] == index), 'x_coords'])[index])
        # source6.data['y_coords'] = ast.literal_eval((contour_data.loc[(contour_data['Cell Name']==signal_select.value) & (contour_data['Time Frame'] == index), 'y_coords'])[index])
        source6.data['x_coords'] = ast.literal_eval((contour_data.loc[(contour_data['Cell Name']==signal_select.value), 'x_coords'])[index])
        source6.data['y_coords'] = ast.literal_eval((contour_data.loc[(contour_data['Cell Name']==signal_select.value), 'y_coords'])[index])
       
    
# Save data function
def save_data():
    selected_signal = signal_select.value
    signal = signals_df[selected_signal]
    prominence_value = prominence_slider.value
    height_value = height_slider.value
    time_of_death = tf_slider.value
    peaks, peak_heights = find_peaks_with_params(signal, prominence_value, height_value)
    data = {
        'Signal': [selected_signal]*len(peaks),
        'Peak pos': peaks,
        'Peak heights': peak_heights,
        'Threshold': [height_value]*len(peaks),
        'Peak proms': [prominence_value]*len(peaks),
        'Time of death': [time_of_death]*len(peaks)
    }
    df = pd.DataFrame(data)
    global peak_data
    peak_data=pd.concat([peak_data, df], ignore_index=True)
    
    # Print or save the DataFrame (adjust as needed)
    print(df)
    
    # Move to the next signal in the dropdown menu
    signal_index = list(signals_df.keys()).index(selected_signal)
    try:
        next_index = (signal_index + 1) % len(signals_df)
        next_signal = list(signals_df.keys())[next_index]
        signal_select.value = next_signal
    except IndexError:
        print('{selected_signal} is the last signal of the dataset'.format(selected_signal=selected_signal))
    
# Callback function to toggle BF display
def callback_display(attr, new, old):
    global display 
    display = not display
    bf_display.visible = display

def callback_contour(attr, new, old):
    global contour
    contour = not contour
    

def tap_point(attr, old, new):
    try:
        
        # peaks = source2.data['peaks']
        # peak_heights = source2.data['heights']
        selected_index = source1.selected.indices[0]
        selected_peak = source1.data['t'][selected_index]
        selected_height = source1.data['signal'][selected_index]
        # add peak if peak is not in the previous peak list
        if selected_peak not in peaks:
            new_peaks = {'peaks': np.append(source2.data['peaks'], selected_peak), 'heights': np.append(source2.data['heights'], selected_height)}
            source2.data = new_peaks
        elif selected_index in peaks:
            global position_selected_height
            global position_selected_peak
            position_selected_height = np.where(source2.data['heights'] == selected_height)[0][0]
            position_selected_peak = np.where(source2.data['peaks'] == selected_peak)[0][0]
            new_peaks = {'peaks': np.delete(source2.data['peaks'], position_selected_peak), 'heights': np.delete(source2.data['heights'], position_selected_height)}
            source2.data = new_peaks
        
    except IndexError:
        pass
        
# Create sliders with on_change callback
prominence_slider = Slider(title='Prominence', value=initial_prominence, start=0.0, end=1.0, step=0.01)
prominence_slider.on_change('value', update_peaks)
height_slider = Slider(title='Height', value=initial_height, start=0.01, end=1.0, step=0.01)
height_slider.on_change('value', update_peaks)
tf_slider = Slider(title='Time of death', value=initial_tf, start=0, end=(len(signal)-1), step=1)
tf_slider.on_change('value', update_tf)

# Create dropdown menu
signal_select = Select(title='Select Signal:', value=selected_signal, options=list(signals_df.keys()))
signal_select.on_change('value', update_signal)

# Create a button to save data
save_button = Button(label="Save Data", button_type="success")
save_button.on_click(save_data)

# Create a checkbox to display BF
checkbox_display = CheckboxGroup(labels=['Display BF'], active=[0,1])
checkbox_display.on_change('active', callback_display)

# Create a checkbox to display contour
checkbox_contour = CheckboxGroup(labels=['Plot contour'], active=[0,1])
checkbox_contour.on_change('active', callback_contour)

# Add or remove peaks by tapping
source1.selected.on_change('indices', tap_point)


# Create a checkbox to consider valid the signal - TO DO



slider_layout = bokeh.layouts.column(
    bokeh.layouts.Spacer(height=30),
    prominence_slider,
    bokeh.layouts.Spacer(height=15),
    height_slider,
    tf_slider,
)

#Dropdown and save button
dropdown_layout = bokeh.layouts.column(
    bokeh.layouts.Spacer(height=30),
    signal_select,
    save_button,
    checkbox_display,
    checkbox_contour
)

# Set up layout
norm_layout = bokeh.layouts.row(
    plot,
    bf_display,
    bokeh.layouts.Spacer(width=15),
    slider_layout,
    dropdown_layout,
)

# Add layout to the current document
def norm_app(doc):
    doc.add_root(norm_layout)

bokeh.io.show(norm_app, notebook_url=notebook_url)

Loading BokehJS ...

FileNotFoundError: [Errno 2] No such file or directory: './contour_data'

In [11]:
len(all_signals_norm)

60

In [22]:
# App to record peaks
# This one implements the time of death and the possibility to hide the display
# Implement manual addition of peaks - removal still to be added
# Implements contour plot
# Implement sliders for beginning of oscillations, end of oscillatiosn
# Implement a button to change the status of the 

import pandas as pd
import numpy as np
from bokeh.plotting import figure, curdoc
from bokeh.models import Button, ColumnDataSource, Slider, Select, Span, CustomJS, CheckboxGroup, ResetTool, RadioGroup, ButtonGroup, TapTool
import bokeh.layouts
import nd2
from pathlib import Path, WindowsPath
from skimage.io import imread
import ast
import scipy

# Turn off warnings
from __future__ import annotations

import logging # isort:skip
log = logging.getLogger(__name__)

from bokeh.util.warnings import BokehUserWarning 
import warnings 
warnings.simplefilter(action='ignore', category=BokehUserWarning)

# Display BF
global display
display = True

global contour
contour = True

# Validity of a given signal
global validity
validity = True

# SInitial values
signals_df = all_signals_norm.copy(deep=True)
# signals_df = df_total.copy(deep=True)
t = np.linspace(0,len(signals_df)-1,len(signals_df))
t = [int(i) for i in t]
selected_signal = signals_df.columns.values[0]
signal = signals_df[selected_signal]

# signal=signal/(np.max(signal))

# Initial parameters for find_peaks
initial_prominence = 0.5
initial_height = 0.01
threshold=np.zeros(len(t))

# Initial parameters for timepoint sliders
initial_tf = 0
v_line = np.linspace(0,len(signals_df)-1, len(signals_df))
tf = np.ones(len(v_line))
bo = np.ones(len(v_line))
eo = np.ones(len(v_line))
# Load and display initial image
image_path = Path('./output/'+selected_signal[:-2]+'.tif')
image_stack = imread(image_path.as_posix())[:,:,:]
img_data={'img':[image_stack[initial_tf]], 'img_stack':[image_stack], 'path':[image_path.as_posix()]}

# Load contour data and plot first one
contour_data = pd.read_csv('./contour_data_ppf021_median_corr', index_col='Time Frame')
contour_dict = {
                'x_coords':ast.literal_eval(contour_data.loc[(contour_data['Cell Name']==selected_signal), 'x_coords'][initial_tf]),
                'y_coords':ast.literal_eval(contour_data.loc[(contour_data['Cell Name']==selected_signal), 'y_coords'][initial_tf]),}

# Data dataframe to store data
global peak_data
peak_data = pd.DataFrame()

# Find Peaks function
def find_peaks_with_params(signal, prominence, height):
    peaks, properties = scipy.signal.find_peaks(signal, prominence=prominence, height=height)
    return peaks, properties['peak_heights']

# Create ColumnDataSource
source1 = ColumnDataSource(data=dict(t=t, signal=signal))                    # Signal and time domain
source2 = ColumnDataSource(data=dict(peaks=[], heights=[]))          # Peaks and peak heights
source3 = ColumnDataSource(data=dict(t=t, threshold=threshold))              # Time domain and height threshold
source4 = ColumnDataSource(data=dict(tf=tf, v_line=v_line))                  # Time frame and vertical line
source5 = ColumnDataSource(data=img_data)                                    # Image data for BF display
source6 = ColumnDataSource(data=contour_dict)                                # Data for contour
source7 = ColumnDataSource(data=dict(bo=bo, v_line=v_line))                  # Beginning of oscillations
source8 = ColumnDataSource(data=dict(eo=eo, v_line=v_line))                  # End of oscillations
source9 = bokeh.models.ColumnDataSource(data=dict(index=[]))  # Data source for the image


# Create Bokeh figure to display signal and peaks
plot = figure(title='normalized  '+signals_df.columns.values[0], x_range=(np.min(t), np.max(t)), y_range=(np.min(signal), np.max(signal)+0.01), width=1000, height=800, tools=[TapTool(), BoxZoomTool(), ResetTool()])
plot.circle('t', 'signal', source=source1, line_width=2, line_color='blue', legend_label='Signal', nonselection_alpha=1.0)
plot.line('t', 'signal', source=source1, line_width=2, line_color='blue', legend_label='Signal')
plot.circle('peaks', 'heights', source=source2, size=8, color='red', legend_label='Peaks', line_width=2, line_color='black', nonselection_alpha=1.0)
plot.line('t', 'threshold', source=source3, line_color='green', line_dash='dashed', legend_label='Height threshold')
tod_renderer = plot.line('tf', 'v_line', source=source4,line_color='red', line_dash='solid', legend_label='Time of death')
plot.line('bo', 'v_line', source=source7,line_color='orange', line_dash='dashed', legend_label='Beginning of oscillations')
plot.line('eo', 'v_line', source=source8,line_color='blue', line_dash='dashed', legend_label='End of oscillations')

# Create Bokeh figure to display BF images
if display == True:
    bf_display = figure(x_range=(0, image_stack.shape[1]), y_range=(0, image_stack.shape[2]))
    #bf_display = figure(x_range=(0, image_stack.shape[1]), y_range=(0, image_stack.shape[2]), width=1000, height=800)
    bf_display.image(image='img', x=0, y=0, dw=image_stack.shape[1], dh=image_stack.shape[2],source=source5, palette='Greys256')
    
    if contour == True:
        contour_renderer = bf_display.line('x_coords', 'y_coords', source=source6,line_color='orange', line_dash='solid', legend_label='Contour')



# Callback function for filtering signal - not implemented for now

# Callback function for dropdown menu
def update_signal(attr, old, new):
    selected_signal = signal_select.value
    signal = signals_df[selected_signal]
    
    # Update data source
    source1.data = dict(t=t, signal=signal)
    
    # Update peaks and heights based on new signal
    prominence_value = prominence_slider.value
    height_value = height_slider.value
    peaks, peak_heights = find_peaks_with_params(signal, prominence_value, height_value)
    source2.data = dict(peaks=peaks, heights=peak_heights)
    
    # Update plot title
    plot.title.text = selected_signal
    
    # Update BF display
    image_path = Path('./output/'+selected_signal[:-2]+'.tif')
    image_stack = imread(image_path.as_posix())[:,:,:]
    img_data={'img':[image_stack[0]], 'img_stack':[image_stack], 'path':[image_path.as_posix()]}
    source5.data = img_data
    tf_slider.value = 0
    
    # Update contour
    if display == True and contour == True:
        source6.data['x_coords'] = ast.literal_eval((contour_data.loc[(contour_data['Cell Name']==signal_select.value), 'x_coords'])[0])
        source6.data['y_coords'] = ast.literal_eval((contour_data.loc[(contour_data['Cell Name']==signal_select.value), 'y_coords'])[0])
    
# Callback function for sliders using on_change
def update_peaks(attr, old, new):
    # Retrieve values from sliders
    prominence_value = prominence_slider.value
    height_value = height_slider.value
    selected_signal = signal_select.value
    signal = signals_df[selected_signal]
    bo = bo_slider.value
    eo = eo_slider.value
    # Update peaks and heights based on new prominence value
    # First use eo as end limit for find peaks, then use boolean array to remove any peak before bo and finally update threshold
    # Caution here: the behaviour of find peaks is different for beginning and end filtering of peaks
    if eo < len(signal): peaks, peak_heights = find_peaks_with_params(signal[:eo+1], prominence_value, height_value)
    else: peaks, peak_heights = find_peaks_with_params(signal, prominence_value, height_value)
    bo_mask = peaks >= bo
    # eo_mask = peaks <= eo
    peaks = peaks[bo_mask]
    peak_heights = peak_heights[bo_mask]
    threshold = np.ones(len(t))*height_value
    # Update data source
    source2.data = dict(peaks=peaks, heights=peak_heights)
    source3.data = dict(t=t, threshold=threshold)

# Callback function for vertical slider
def update_tf(attr, old, new):
    
   # Neccessary a check to change the signal only if a different signal in the dropdown menu is selected!!!
   # Update line in signal plot
    index = tf_slider.value
    tf = np.ones(len(v_line)) * index
    source4.data = dict(tf=tf, v_line=v_line)
    selected_time_lapse = signal_select.value
    image_path = Path('./output/'+selected_time_lapse[:-2]+'.tif').as_posix()
    if image_path != source5.data['path'][0]:
        image_path = Path('./output/'+selected_image[:-2]+'.tif')
        image_stack = nd2.imread(image_path.as_posix())[:,:,:]
    else:
        image_stack = source5.data['img_stack'][0]
    
    # Update tf and contour in BF display
    if display == True:
        new_image = image_stack[index]
        source5.data['img'] = [new_image]
        if contour == True:
            source6.data['x_coords'] = ast.literal_eval((contour_data.loc[(contour_data['Cell Name']==signal_select.value), 'x_coords'])[index])
            source6.data['y_coords'] = ast.literal_eval((contour_data.loc[(contour_data['Cell Name']==signal_select.value), 'y_coords'])[index])


# Callback for beginning of oscillations
def update_bo(attr, old, new):
    index = bo_slider.value
    bo = np.ones(len(v_line)) * index
    source7.data = dict(bo=bo, v_line=v_line)
    
# Callback for end of oscillations
def update_eo(attr, old, new):
    index = eo_slider.value
    eo = np.ones(len(v_line)) * index
    source8.data = dict(eo=eo, v_line=v_line)
    
# Save data function
def save_data():
    selected_signal = signal_select.value
    signal = signals_df[selected_signal]
    prominence_value = prominence_slider.value
    height_value = height_slider.value
    time_of_death = tf_slider.value
    bo = bo_slider.value
    eo = eo_slider.value
    peaks, peak_heights = find_peaks_with_params(signal, prominence_value, height_value)
    
    global peak_data
    try:
        if peak_data['Signal'].isin([selected_signal]).any(): # Remove the stored-data if the signal is re-done
            peak_data = peak_data[~peak_data['Signal'].isin([selected_signal])]
    
    except KeyError:
        print('First tp')
        
    if validity == True:
        data = {
            'Signal': [selected_signal]*len(peaks),
            'Peak pos': peaks,
            'Peak heights': peak_heights,
            'Threshold': [height_value]*len(peaks),
            'Peak proms': [prominence_value]*len(peaks),
            'Time of death': [time_of_death]*len(peaks),
            'Beginning of oscillation': [bo] *len(peaks),
            'End of oscillation': [eo]*len(peaks)
        }
    else:
        data = {
            'Signal': [selected_signal],
            'Peak pos': np.nan,
            'Peak heights': np.nan,
            'Threshold': np.nan,
            'Peak proms': np.nan,
            'Time of death': np.nan,
            'Beginning of oscillation': np.nan,
            'End of oscillation': np.nan
        }
    if display == False:
        data['Time of death'] = np.nan
    df = pd.DataFrame(data)
    peak_data=pd.concat([peak_data, df], ignore_index=True)
    
    # Print or save the DataFrame (adjust as needed)
    print(df)
    
    # Move to the next signal in the dropdown menu
    signal_index = list(signals_df.keys()).index(selected_signal)
    try:
        # next_index = (signal_index + 1) % len(signals_df)
        next_index = signal_index + 1
        next_signal = list(signals_df.keys())[next_index]
        signal_select.value = next_signal
    except IndexError:
        print('{selected_signal} is the last signal of the dataset'.format(selected_signal=selected_signal))
    
# Callback function to toggle BF display
def callback_display(attr, new, old):
    global display 
    display = not display
    bf_display.visible = display
    tf_slider.visible = display
    checkbox_contour.visible = display
    tod_renderer.visible = display

def callback_contour(attr, new, old):
    global contour
    contour = not contour
    contour_renderer.visible = contour
    

    
def select_tap_callback():
    return """
    const indices = cb_data.source.selected.indices;

    if (indices.length > 0) {
        const index = indices[0];
        other_source.data = {'index': [index]};
        other_source.change.emit();  
    }
    """
def remove_peak(attr, old, new):
    try:
        # peaks = source2.data['peaks']
        # peak_heights = source2.data['heights']
        selected_index = int(new['index'][0])
        selected_peak = source1.data['t'][selected_index]
        selected_height = source1.data['signal'][selected_index]
        # add peak if peak is not in the previous peak list
        print('**************')
        print(selected_index)
        print(selected_peak)
        print(selected_height)
        print(source2.data['peaks'])
        print('-------------------')
        peak_list=[int(peak) for peak in source2.data['peaks']]
        height_list=[int(peak) for peak in source2.data['heights']] 

        print(peak_list)
        if selected_peak not in peak_list:
            new_peaks = {'peaks': peak_list.append(selected_peak), 'heights': height_list.append(selected_height)}
            source2.data = new_peaks
        else:
            temp_index = source2.data['peaks'].tolist().index(selected_index)
            temp_height = source2.data['heights'].tolist().remove(source2.data['heights'][temp_index])
            temp_peaks = source2.data['peaks'].tolist().remove(selected_index)
            print(temp_index)
            print(temp_height)
            print(temp_peaks)
            if temp_height == None and temp_peaks==None: 
                temp_height=[]
                temp_peaks=[]
            new_peaks = {'peaks':temp_peaks, 'heights': temp_height}
            source2.data = new_peaks
        
    except IndexError:
        pass


def tap_point(attr, old, new):
    try:
        print(peaks)
        print(source2.data['peaks'])
        # peak_heights = source2.data['heights']
        selected_index = source1.selected.indices[0]
        selected_peak = source1.data['t'][selected_index]
        selected_height = source1.data['signal'][selected_index]
        # add peak if peak is not in the previous peak list
        if selected_peak not in peaks:
            new_peaks = {'peaks': np.append(source2.data['peaks'], selected_peak), 'heights': np.append(source2.data['heights'], selected_height)}
            source2.data = new_peaks
        elif selected_index in peaks:
            global position_selected_height
            global position_selected_peak
            position_selected_height = np.where(source2.data['heights'] == selected_height)[0][0]
            position_selected_peak = np.where(source2.data['peaks'] == selected_peak)[0][0]
            new_peaks = {'peaks': np.delete(source2.data['peaks'], position_selected_peak), 'heights': np.delete(source2.data['heights'], position_selected_height)}
            source2.data = new_peaks
        
    except IndexError:
        pass
    

def change_validity(new):
    global validity
    validity = not validity

# Create sliders with on_change callback
prominence_slider = Slider(title='Prominence', value=initial_prominence, start=0.0, end=1.0, step=0.01)
prominence_slider.on_change('value', update_peaks)
height_slider = Slider(title='Height', value=initial_height, start=0.01, end=1.0, step=0.01)
height_slider.on_change('value', update_peaks)
tf_slider = Slider(title='Time of death', value=initial_tf, start=0, end=(len(signal)-1), step=1)
tf_slider.on_change('value', update_tf)
bo_slider = Slider(title='Beginning of oscillations', value=initial_tf, start=0, end=(len(signal)-1), step=1)
bo_slider.on_change('value', update_bo)
bo_slider.on_change('value', update_peaks)
eo_slider = Slider(title='End of oscillations', value=initial_tf, start=0, end=(len(signal)-1), step=1)
eo_slider.on_change('value', update_eo)
eo_slider.on_change('value', update_peaks)


# Create dropdown menu
signal_select = Select(title='Select Signal:', value=selected_signal, options=list(signals_df.keys()))
signal_select.on_change('value', update_signal)

# Create a button to save data
save_button = Button(label="Save Data", button_type="success")
save_button.on_click(save_data)

# Create a radiobutton to change validity of data
validity_button = RadioGroup(labels=['Valid', 'Not Valid'], active=0)
validity_button.on_click(change_validity)



# Create a checkbox to display BF
checkbox_display = CheckboxGroup(labels=['Display BF'], active=[0,1])
checkbox_display.on_change('active', callback_display)

# Create a checkbox to display contour
checkbox_contour = CheckboxGroup(labels=['Plot contour'], active=[0,1])
checkbox_contour.on_change('active', callback_contour)

# Add or remove peaks by tapping
# source1.selected.on_change('indices', tap_point)

tap_tool = bokeh.models.TapTool(callback=bokeh.models.CustomJS(args=dict(other_source=source9),code=select_tap_callback()))



plot.add_tools(tap_tool)
source9.on_change('data', remove_peak)



slider_layout = bokeh.layouts.column(
    bokeh.layouts.Spacer(height=30),
    prominence_slider,
    bokeh.layouts.Spacer(height=15),
    height_slider,
    tf_slider,
    bo_slider,
    eo_slider
)

#Dropdown and save button
dropdown_layout = bokeh.layouts.column(
    bokeh.layouts.Spacer(height=30),
    signal_select,
    save_button,
    checkbox_display,
    checkbox_contour,
    validity_button
)

# Set up layout
norm_layout = bokeh.layouts.row(
    plot,
    bf_display,
    bokeh.layouts.Spacer(width=15),
    slider_layout,
    dropdown_layout,
)
notebook_url = 'localhost:8888'
# Add layout to the current document
def norm_app(doc):
    doc.add_root(norm_layout)

bokeh.io.show(norm_app, notebook_url=notebook_url)

INFO:bokeh.server.server:Starting Bokeh server version 2.4.3 (running on Tornado 6.4)
INFO:bokeh.server.tornado:User authentication hooks NOT provided (default user enabled)
C:\Users\pperez\.conda\envs\devbio-napari-cupy\lib\site-packages\bokeh\io\notebook.py:487: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  publish_display_data(data, metadata, source, transient=transient, **kwargs)


INFO:tornado.access:200 GET /autoload.js?bokeh-autoload-element=1273&bokeh-absolute-url=http://localhost:59303&resources=none (::1) 62.00ms
INFO:tornado.access:101 GET /ws (::1) 1.00ms
INFO:bokeh.server.views.ws:WebSocket connection opened
INFO:bokeh.server.views.ws:ServerConnection created


First tp
           Signal  Peak pos  Peak heights  Threshold  Peak proms  \
0  ppf021_xy001_0         3      0.583115       0.01         0.1   
1  ppf021_xy001_0        10      1.000000       0.01         0.1   

   Time of death  Beginning of oscillation  End of oscillation  
0              0                         0                  16  
1              0                         0                  16  
           Signal  Peak pos  Peak heights  Threshold  Peak proms  \
0  ppf021_xy004_0         2      0.204797       0.01         0.1   
1  ppf021_xy004_0         9      0.397968       0.01         0.1   
2  ppf021_xy004_0        16      1.000000       0.01         0.1   

   Time of death  Beginning of oscillation  End of oscillation  
0              0                         0                  23  
1              0                         0                  23  
2              0                         0                  23  
           Signal  Peak pos  Peak heights  Threshold  Peak

ERROR:bokeh.server.protocol_handler:error handling message
 message: Message 'PATCH-DOC' content: {'events': [{'kind': 'MessageSent', 'msg_type': 'bokeh_event', 'msg_data': {'event_name': 'button_click', 'event_values': {}}}], 'references': []} 
 error: KeyError(0)
Traceback (most recent call last):
  File "C:\Users\pperez\.conda\envs\devbio-napari-cupy\lib\site-packages\pandas\core\indexes\base.py", line 3803, in get_loc
    return self._engine.get_loc(casted_key)
  File "pandas\_libs\index.pyx", line 138, in pandas._libs.index.IndexEngine.get_loc
  File "pandas\_libs\index.pyx", line 165, in pandas._libs.index.IndexEngine.get_loc
  File "pandas\_libs\hashtable_class_helper.pxi", line 2263, in pandas._libs.hashtable.Int64HashTable.get_item
  File "pandas\_libs\hashtable_class_helper.pxi", line 2273, in pandas._libs.hashtable.Int64HashTable.get_item
KeyError: 0

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "C:\Users\ppe

            Signal  Peak pos  Peak heights  Threshold  Peak proms  \
0   ppf021_xy108_0         7      0.914405       0.01        0.01   
1   ppf021_xy108_0        17      1.000000       0.01        0.01   
2   ppf021_xy108_0        24      0.030340       0.01        0.01   
3   ppf021_xy108_0        31      0.049343       0.01        0.01   
4   ppf021_xy108_0        35      0.069247       0.01        0.01   
5   ppf021_xy108_0        40      0.063642       0.01        0.01   
6   ppf021_xy108_0        43      0.087422       0.01        0.01   
7   ppf021_xy108_0        48      0.063967       0.01        0.01   
8   ppf021_xy108_0        55      0.082820       0.01        0.01   
9   ppf021_xy108_0        60      0.077321       0.01        0.01   
10  ppf021_xy108_0        66      0.080779       0.01        0.01   
11  ppf021_xy108_0        69      0.081571       0.01        0.01   
12  ppf021_xy108_0        72      0.080035       0.01        0.01   

    Time of death  Beginning of o

C:\Users\pperez\.conda\envs\devbio-napari-cupy\lib\site-packages\bokeh\server\protocol_handler.py:101: RuntimeWarning: coroutine 'WSHandler.send_message' was never awaited
  work = connection.error(message, repr(e))


            Signal  Peak pos  Peak heights  Threshold  Peak proms  \
0   ppf021_xy108_1         2      0.038964       0.01        0.01   
1   ppf021_xy108_1         5      0.078770       0.01        0.01   
2   ppf021_xy108_1        11      0.042403       0.01        0.01   
3   ppf021_xy108_1        19      0.198719       0.01        0.01   
4   ppf021_xy108_1        32      1.000000       0.01        0.01   
5   ppf021_xy108_1        35      0.650102       0.01        0.01   
6   ppf021_xy108_1        52      0.025923       0.01        0.01   
7   ppf021_xy108_1        54      0.040769       0.01        0.01   
8   ppf021_xy108_1        57      0.027372       0.01        0.01   
9   ppf021_xy108_1        59      0.031324       0.01        0.01   
10  ppf021_xy108_1        64      0.021364       0.01        0.01   
11  ppf021_xy108_1        69      0.029558       0.01        0.01   
12  ppf021_xy108_1        71      0.022473       0.01        0.01   

    Time of death  Beginning of o

In [19]:
peak_data

,Signal,Peak pos,Peak heights,Threshold,Peak proms,Time of death,Beginning of oscillation,End of oscillation
0,ppf021_xy001_0,3.0,0.510376,0.01,0.21,21.0,0.0,15.0
1,ppf021_xy001_0,10.0,1.000000,0.01,0.21,21.0,0.0,15.0
2,ppf021_xy004_0,2.0,0.298873,0.01,0.03,0.0,0.0,23.0
3,ppf021_xy004_0,8.0,0.528146,0.01,0.03,0.0,0.0,23.0
4,ppf021_xy004_0,16.0,1.000000,0.01,0.03,0.0,0.0,23.0
...,...,...,...,...,...,...,...,...
446,ppf021_xy108_0,57.0,0.134768,0.01,0.03,0.0,0.0,23.0
447,ppf021_xy108_0,59.0,0.139288,0.01,0.03,0.0,0.0,23.0
448,ppf021_xy108_0,64.0,0.133710,0.01,0.03,0.0,0.0,23.0
449,ppf021_xy108_0,66.0,0.104965,0.01,0.03,0.0,0.0,23.0


In [23]:
duplicate = peak_data.copy(deep=True)

In [24]:
duplicate = duplicate.dropna(subset=['Peak pos'])

In [25]:
def trim_function (peak_data=duplicate):
    trimmed = pd.DataFrame()
    l_trimmed = []

    for signal in peak_data['Signal'].unique():
        bo = duplicate.loc[peak_data['Signal'] == signal]['Beginning of oscillation'].min()
        eo = duplicate.loc[peak_data['Signal'] == signal]['End of oscillation'].max()
        l_trimmed.append(peak_data.loc[(peak_data['Signal'] == signal) & (peak_data['Peak pos'] > bo) & (peak_data['Peak pos'] < eo), :])
    
    trimmed=pd.concat(l_trimmed)
    return trimmed

In [26]:
trimmed = pd.DataFrame()
l_trimmed = []

for signal in duplicate['Signal'].unique():
    bo = duplicate.loc[duplicate['Signal'] == signal]['Beginning of oscillation'].min()
    eo = duplicate.loc[duplicate['Signal'] == signal]['End of oscillation'].max()
    l_trimmed.append(duplicate.loc[(duplicate['Signal'] == signal) & (duplicate['Peak pos'] > bo) & (duplicate['Peak pos'] < eo), :])

In [27]:
trimmed = pd.concat(l_trimmed).reset_index()

In [28]:
trimmed

,index,Signal,Peak pos,Peak heights,Threshold,Peak proms,Time of death,Beginning of oscillation,End of oscillation
0,0,ppf021_xy001_0,3.0,0.583115,0.01,0.10,0.0,0.0,16.0
1,1,ppf021_xy001_0,10.0,1.000000,0.01,0.10,0.0,0.0,16.0
2,2,ppf021_xy004_0,2.0,0.204797,0.01,0.10,0.0,0.0,23.0
3,3,ppf021_xy004_0,9.0,0.397968,0.01,0.10,0.0,0.0,23.0
4,4,ppf021_xy004_0,16.0,1.000000,0.01,0.10,0.0,0.0,23.0
...,...,...,...,...,...,...,...,...,...
263,543,ppf021_xy108_0,17.0,1.000000,0.01,0.01,0.0,0.0,24.0
264,555,ppf021_xy108_1,2.0,0.038964,0.01,0.01,0.0,0.0,24.0
265,556,ppf021_xy108_1,5.0,0.078770,0.01,0.01,0.0,0.0,24.0
266,557,ppf021_xy108_1,11.0,0.042403,0.01,0.01,0.0,0.0,24.0


In [29]:
trimmed.drop('index',axis=1,inplace=True)

In [30]:
trimmed.to_csv('./peak_data_ppf021_median_corr')

In [91]:
trimmed = pd.DataFrame()
l_trimmed = []

In [92]:
for signal in duplicate['Signal'].unique():
    bo = duplicate.loc[duplicate['Signal'] == signal]['Beginning of oscillation'].min()
    eo = duplicate.loc[duplicate['Signal'] == signal]['End of oscillation'].max()
    l_trimmed.append(duplicate.loc[(duplicate['Signal'] == signal) & (duplicate['Peak pos'] > bo) & (duplicate['Peak pos'] < eo), :])


In [36]:
import re
def assign_condition(signal):
    # Extract nnn using regex
    match = re.search(r'_xy(\d+)_', signal)
    if match:
        nnn = int(match.group(1))
        # Define conditions based on nnn value
        if nnn <= 35:
            return 'Control'
        elif 36 <= nnn <= 75:
            return '40 nM'
        elif 76 <= nnn <= 110:
            return '80 nM'
        else:
            return 'Unknown'
    return 'Unknown'

# Apply the function to create the Condition column
trimmed['Condition'] = trimmed['Signal'].apply(assign_condition)

In [37]:
trimmed.to_csv('./peak_data_ppf021_median_corr')

In [38]:
# Group by 'Signal' and count unique 'Peak pos' values
collapsed_df = trimmed.groupby('Signal').agg(
    Peak_pos_count=('Peak pos', 'nunique'),
    Beginning_of_oscillation=('Beginning of oscillation', 'first'),
    End_of_oscillation=('End of oscillation', 'first'),
    Condition=('Condition', 'first')
).reset_index()

In [39]:
collapsed_df['Osc_time'] = collapsed_df['End_of_oscillation'] - collapsed_df['Beginning_of_oscillation']

In [40]:
import pandas as pd
import numpy as np
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.layouts import column
from bokeh.models import ColumnDataSource
from bokeh.transform import factor_cmap
from bokeh.palettes import d3
# from bokeh.io import 


# Group by 'Condition'
grouped = collapsed_df.groupby('Condition')

# Prepare output
output_notebook()

# Select a categorical color palette from D3
palette = d3['Category10'][4]  # Adjust the number here based on the number of unique conditions

# Create figures for each condition
plots = []
conditions = collapsed_df['Condition'].unique()

for i, (name, group) in enumerate(grouped):
    
    hist, edges = np.histogram(group['Peak_pos_count'], bins=range(1, max(collapsed_df['Peak_pos_count']) + 2))
    source = ColumnDataSource(data={'top': hist, 'left': edges[:-1], 'right': edges[1:], 'Condition': [name] * len(hist)})
    
    num_signals = len(group)
    
    # Assign name
    if name == 'Control':
        p = figure(title=f"Exp:ppf020 - Condition: {name}", x_axis_label='Peak pos count', y_axis_label='Hits', tools='pan,wheel_zoom,box_zoom,reset,save')
    else:
        p = figure(title=f"Exp:ppf020 - Condition: {name} CHX", x_axis_label='Peak pos count', y_axis_label='Hits', tools='pan,wheel_zoom,box_zoom,reset,save')

    p.quad(top='top', bottom=0, left='left', right='right', source=source, fill_color=palette[i], line_color='black', alpha=0.5)
    
    # Stats
    
    collapsed_mean = collapsed_df.groupby('Condition')['Peak_pos_count'].agg('mean').reset_index()
    collapsed_median = collapsed_df.groupby('Condition')['Peak_pos_count'].agg('median').reset_index()
    collapsed_min = collapsed_df.groupby('Condition')['Peak_pos_count'].agg('min').reset_index()
    collapsed_max = collapsed_df.groupby('Condition')['Peak_pos_count'].agg('max').reset_index()
    collapsed_std = collapsed_df.groupby('Condition')['Peak_pos_count'].agg(np.std, ddof=1).reset_index()



    
    mean = collapsed_mean.loc[collapsed_mean['Condition']==name, 'Peak_pos_count'].to_numpy().item()
    median = collapsed_median.loc[collapsed_median['Condition']==name, 'Peak_pos_count'].to_numpy().item()
    min_peak = collapsed_min.loc[collapsed_min['Condition']==name, 'Peak_pos_count'].to_numpy().item()
    max_peak = collapsed_max.loc[collapsed_max['Condition']==name, 'Peak_pos_count'].to_numpy().item()
    std = collapsed_std.loc[collapsed_max['Condition']==name, 'Peak_pos_count'].to_numpy().item()
    

    
    p.circle([], [], legend_label=f"Total signals: {num_signals}", fill_color=palette[i])
    p.circle([], [], legend_label=f"Mean: {mean:.2f}", fill_color=palette[i])
    p.circle([], [], legend_label=f"Std: {std:.2f}", fill_color=palette[i])
    p.circle([], [], legend_label=f"Median: {median:.2f}", fill_color=palette[i])
    p.circle([], [], legend_label=f"Min peak: {min_peak:.0f}", fill_color=palette[i])
    p.circle([], [], legend_label=f"Max peak: {max_peak:.0f}", fill_color=palette[i])


    
    p.legend.title = 'Statistics'
    p.legend.location = 'top_right'
    p.legend.label_text_font_size = '8pt'
    p.xaxis.axis_label = 'Number of oscillations'
    p.xaxis.axis_label_text_font_size='12pt'
    p.output_backend = "svg"
    # export_svgs(p, filename=f"./images/{name}_n_oscillations.svg")
    plots.append(p)


# Arrange plots in a column
layout = column(*plots)

# Show plot
show(layout)

C:\Users\pperez\.conda\envs\devbio-napari-cupy\lib\site-packages\bokeh\io\notebook.py:487: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  publish_display_data(data, metadata, source, transient=transient, **kwargs)


Loading BokehJS ...

C:\Users\pperez\.conda\envs\devbio-napari-cupy\lib\site-packages\bokeh\io\notebook.py:487: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  publish_display_data(data, metadata, source, transient=transient, **kwargs)


In [41]:
import pandas as pd
import numpy as np
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.layouts import column
from bokeh.models import ColumnDataSource
from bokeh.transform import factor_cmap
from bokeh.palettes import d3
from bokeh.io import export_svgs


# Group by 'Condition'
grouped = collapsed_df.groupby('Condition')

# Prepare output
output_notebook()

# Select a categorical color palette from D3
palette = d3['Category10'][5]  # Adjust the number here based on the number of unique conditions

# Create figures for each condition
plots = []
conditions = collapsed_df['Condition'].unique()

for i, (name, group) in enumerate(grouped):
    # bins = numpy.histogram_bin_edges(collapsed_2df['Osc_time'], bins=fd, range=None, weights=None)
    hist, edges = np.histogram(group['Osc_time'] * 15, bins='fd')
    source = ColumnDataSource(data={'top': hist, 'left': edges[:-1], 'right': edges[1:], 'Condition': [name] * len(hist)})
    
    num_signals = len(group)
    
    # Assign name
    if name == 'Control':
        p = figure(title=f"Exp:ppf008 - Condition: {name}", x_axis_label='Oscillation time', y_axis_label='Hits', tools='pan,wheel_zoom,box_zoom,reset,save')
    else:
        p = figure(title=f"Exp:ppf008 - Condition: {name} a-amanitin", x_axis_label='Oscillation time', y_axis_label='Hits', tools='pan,wheel_zoom,box_zoom,reset,save')

    p.quad(top='top', bottom=0, left='left', right='right', source=source, fill_color=palette[i], line_color='black', alpha=0.5)
    
    # Stats
    
    collapsed_mean = collapsed_df.groupby('Condition')['Osc_time'].agg('mean').reset_index()
    collapsed_median = collapsed_df.groupby('Condition')['Osc_time'].agg('median').reset_index()
    collapsed_min = collapsed_df.groupby('Condition')['Osc_time'].agg('min').reset_index()
    collapsed_max = collapsed_df.groupby('Condition')['Osc_time'].agg('max').reset_index()
    collapsed_std = collapsed_df.groupby('Condition')['Osc_time'].agg(np.std, ddof=1).reset_index()



    
    mean = collapsed_mean.loc[collapsed_mean['Condition']==name, 'Osc_time'].to_numpy().item() * 12.5
    median = collapsed_median.loc[collapsed_median['Condition']==name, 'Osc_time'].to_numpy().item() *12.5 
    min_time = collapsed_min.loc[collapsed_min['Condition']==name, 'Osc_time'].to_numpy().item()*12.5
    max_time = collapsed_max.loc[collapsed_max['Condition']==name, 'Osc_time'].to_numpy().item()*12.5
    std_time = collapsed_std.loc[collapsed_max['Condition']==name, 'Osc_time'].to_numpy().item()*12.5

    

    
    p.circle([], [], legend_label=f"Total signals: {num_signals}", fill_color=palette[i])
    p.circle([], [], legend_label=f"Mean osc time: {mean:.2f}", fill_color=palette[i])
    p.circle([], [], legend_label=f"Std osc time: {std_time:.2f}", fill_color=palette[i])
    p.circle([], [], legend_label=f"Median osc time: {median:.2f}", fill_color=palette[i])
    p.circle([], [], legend_label=f"Min osc time: {min_time:.0f}", fill_color=palette[i])
    p.circle([], [], legend_label=f"Max osc time: {max_time:.0f}", fill_color=palette[i])


    
    p.legend.title = 'Statistics'
    p.legend.location = 'top_right'
    p.legend.label_text_font_size = '8pt'
    p.xaxis.axis_label = 'Oscillation time (min)'
    p.xaxis.axis_label_text_font_size='12pt'
    
    plots.append(p)
    p.output_backend = "svg"
    # export_svgs(p, filename=f"./images/{name}_amanitin_osc_time.svg")
# Arrange plots in a column
layout = column(*plots)

# Show plot
show(layout)

Loading BokehJS ...

C:\Users\pperez\.conda\envs\devbio-napari-cupy\lib\site-packages\bokeh\io\notebook.py:487: DeprecationWarning: The `source` parameter emit a  deprecation warning since IPython 8.0, it had no effects for a long time and will  be removed in future versions.
  publish_display_data(data, metadata, source, transient=transient, **kwargs)


In [29]:
trimmed[]

SyntaxError: invalid syntax (624883311.py, line 1)

In [34]:
collapsed_df.groupby('Condition')['Peak_pos_count'].agg('mean').reset_index()

,Condition,Peak_pos_count
0,10 nM,4.944444
1,100 nM,3.846154
2,1000 nM,5.250000
3,Control,3.750000


In [30]:
collapsed_df.groupby('Condition')['Peak_pos_count'].agg('median').reset_index()

,Condition,Peak_pos_count
0,10 nM,5.0
1,100 nM,4.0
2,1000 nM,5.0
3,Control,4.0


In [31]:
collapsed_df.groupby('Condition')['Peak_pos_count'].agg('min').reset_index()

,Condition,Peak_pos_count
0,10 nM,2
1,100 nM,2
2,1000 nM,4
3,Control,2


In [32]:
collapsed_df.groupby('Condition')['Peak_pos_count'].agg('max').reset_index()

,Condition,Peak_pos_count
0,10 nM,8
1,100 nM,6
2,1000 nM,6
3,Control,5


In [33]:
for i, x in enumerate(grouped):
    print(i, x)

0 ('10 nM',             Signal  Peak_pos_count  Beginning_of_oscillation  \
16  ppf020_xy032_0               3                       0.0   
17  ppf020_xy033_0               7                       0.0   
18  ppf020_xy034_0               6                       0.0   
19  ppf020_xy035_0               6                      10.0   
20  ppf020_xy037_0               5                       0.0   
21  ppf020_xy038_0               5                       0.0   
22  ppf020_xy039_0               4                       0.0   
23  ppf020_xy040_0               7                       0.0   
24  ppf020_xy041_0               4                       0.0   
25  ppf020_xy042_0               4                       0.0   
26  ppf020_xy045_0               4                       8.0   
27  ppf020_xy049_0               2                       0.0   
28  ppf020_xy049_1               2                       0.0   
29  ppf020_xy050_0               3                       0.0   
30  ppf020_xy053_0          

In [ ]:
collapsed_2 = 